# GameTheory-16e : Populations Othman-Sandholm — généralisation du pilote 16d

Ce side-car **étend** le pilote `GameTheory-16d-LLM-Players-Othman-Sandholm-Python.ipynb` : au lieu de jouer **une seule paire** (row, col) sur le
mécanisme de la Proposition 6 d'Othman & Sandholm (SAGT 2009, *Better with
Byzantine*), on joue une **population** de paires — un échantillon indépendant de
N paires tirées selon une distribution sur les politiques.

**Déconflit** : ce carnet ne réécrit pas le 16d (pilote fondateur) ni les
16/16b/16c (mechanism design). Il **re-gèle** le protocole §0 sur des
populations au lieu d'une paire.

**Périmètre v1** : quatre populations scriptées (uniform_random / always_a_prime / always_a /
byzantine), cinq seeds parmi {0, 1, 7, 42, 99}, modèle nul explicite
(uniform_random), contrôle négatif (politique `always_a` qui dégrade le
welfare). **Périmètre v2 (§10)** : deux populations mixtes LLM × scripts
(`mix-dominant`, `mix-byzantin`), rejouées en C# hors ligne à partir de données
citées du jumeau Python (glm-5.2 ; qwen3.8-max exclu par 429 persistants,
politique réseau gelée).

**Protocole** :
- Protocole re-gelé avant le premier appel de stratégie (cf §0, et §10.0 pour la v2).
- ≥ 2 populations hétérogènes : 4 scriptées (v1, §2) + 2 mixtes LLM × scripts
  (v2, §10).
- Multi-seed ≥ 4 sur les seeds {0, 1, 7, 42, 99}.
- Modèle nul nommé et contrôle négatif rapporté.
- Verdicts H0/H1 par population, lecture des seuils dans le gel §0.

**Navigation** : [Index GameTheory](README.md) | [Version Python](GameTheory-16e-Populations-Othman-Sandholm-Python.ipynb) | Socle : [GameTheory-16 -- Conception de mecanismes](GameTheory-16-MechanismDesign-Python.ipynb) (§4.6, Proposition 6 d'Othman-Sandholm) | Pilote : [GameTheory-16d -- Joueurs LLM hétérogènes](GameTheory-16d-LLM-Players-Othman-Sandholm-Python.ipynb) | Extension : [GameTheory-16h -- Coopération LLM](GameTheory-16h-Cooperation-LLM-Python.ipynb) (protocole multi-seed IPD).

In [1]:
// Préambule — paramètres et outils de vérification
using System;
using System.Collections.Generic;
using System.Globalization;
using System.Linq;
using System.Text.Json;

var SEEDS = Array.AsReadOnly(new[] { 0, 1, 7, 42, 99 });
const int N_PAIRS = 50; // 50 paires par seed (gel §0)

// Les seuils expérimentaux ne sont pas des tolérances de parité.
const double TOL_EXACTE = 0.0; // Entiers et constantes identiques dans les deux langages.
const double TOL_FLOTTANTE = 1e-12; // Marge pour les opérations en double sur de petits échantillons.
var ecartsParSection = new Dictionary<string, List<(string Mesure, double Calcule, double Cite, double Ecart, double Tolerance)>>(StringComparer.Ordinal);

string Inv(FormattableString texte) => texte.ToString(CultureInfo.InvariantCulture);

void Exiger(bool condition, string message)
{
    if (!condition)
        throw new InvalidOperationException(message);
}

void VerifierEcart(string section, string mesure, double calcule, double cite, double tolerance)
{
    double ecart = Math.Abs(calcule - cite);
    Exiger(ecart <= tolerance,
        Inv($"ÉCART {section} — {mesure} : calculé={calcule:G17}, attendu={cite:G17}, écart={ecart:G17}, tolérance={tolerance:G17}"));
    if (!ecartsParSection.TryGetValue(section, out var mesures))
    {
        mesures = new List<(string Mesure, double Calcule, double Cite, double Ecart, double Tolerance)>();
        ecartsParSection.Add(section, mesures);
    }
    mesures.Add((mesure, calcule, cite, ecart, tolerance));
}

// Une référence publiée à d décimales est connue à une demi-unité du dernier chiffre près.
double TolerancePubliee(int decimales) => 0.5 * Math.Pow(10.0, -decimales) + TOL_FLOTTANTE;

Exiger(SEEDS.Count >= 4, "Multi-seed protocol : >= 4 seeds parmi {0,1,7,42,99} (gel §0).");
Exiger(N_PAIRS >= 20, "Population size : N_PAIRS >= 20 pour stabilite des estimateurs.");

## Préambule — qu'est-ce qu'une *population* ?

Le pilote 16d joue **une paire** (row, col) sur le mécanisme M : deux rapports
de type (`a` ou `a'`), un outcome (`o1..o4`) produit par M, et un oracle de
bien-être social `sw_oracle(outcome, theta_row, theta_col)` qui somme les
utilités.

Quand on parle de **populations**, on joue **N paires indépendantes** selon
une *politique* commune. Chaque paire pioche `(theta_row, theta_col)` parmi
les profils vrais (`(a,a)`, `(a,a')`, `(a',a)`, `(a',a')`) — avec une
distribution fixée par la **politique** (par ex. uniform_random, équilibré,
adverse, etc.) — puis chaque rôle joue sa politique d'action (`r_row`,
`r_col`) à partir de l'information visible par rôle (alpha = section 1).

La métrique d'une population est **la distribution de sw_oracle sur N paires** :
moyenne, médiane, écart-type, comptes par profil vrai, comptes par outcome.

Hétérogénéité = différence de politique entre populations ; une population
`uniform_random` (modèle nul) sert de baseline inerte.**Trois populations
scriptées** (`random`, `always_a'`, `always_a`) sont comparées au nul, et une
**population byzantine** sert de contrôle négatif (perturbe le welfare).

## 0. Pré-enregistrement du protocole (gelé avant tout appel de stratégie)

Rôles, conditions, profils vrais, populations, seeds, hypothèses, seuils de
verdict et contrôles sont figés ici. Toute déviation exécution vs gel est
consignée au §9 (Écarts d'exécution vs gel) comme violation de protocole.

**Rôles** : `row` (joueur 1, observe `r_col` adverse) et `col` (joueur 2,
observe `r_row` adverse). Information symétrique par condition.

**Conditions (4 profils vrais)** : `(a,a)`, `(a,a')`, `(a',a)`, `(a',a')` —
probabilités **uniformes** sur les 4 profils (modèle nul = uniform_random).

**Populations (4, hétérogènes)** :
- `uniform_random` | chaque action pioche uniformément parmi `{a, a'}` (modèle nul)
- `always_a_prime` | chaque action rapporte `a'` (dominant en action, cf 16d §4.6)
- `always_a` | chaque action rapporte `a` (perdent sous l'oracle M)
- `byzantine` | chaque action rapporte le **complément** de son type vrai (perturbation)

**Politique d'information** : un agent observe uniquement le rapport de
l'adversaire, pas son propre type vrai — c'est la formulation canonique
mechanism design (sans signal privé au joueur).

**Seeds** : `{0, 1, 7, 42, 99}` — un sous-ensemble strict de 4+ éléments du
jeu standard (cf registre pilote 16d). 5 seeds pour la robustesse.

**Taille de population** : `N_PAIRS = 50` paires indépendantes par seed — borne
basse pour stabiliser moyenne/écart-type sur 4 profils, sans surcoût.

**Métriques** :
- `sw_optimal(theta_row, theta_col)` (oracle, lecture §1)
- `sw_oracle(outcome, theta_row, theta_col)` (mêmes chiffres, sanity check)
- Distribution de sw par population × seed : moyenne, médiane, écart-type.

**Seuils de verdict (H0/H1)** :
- H0 — *toutes populations produisent le même welfare moyen que le nul* —
  test : moyenne par seed dans ±0.1 du nul.
- H1 — *au moins une population s'écarte significativement du nul* — sinon.

**Modèle nul** : `uniform_random` (4 profils équiprobables × 2 actions
polaires équiprobables) — c'est la distribution uniforme sur les 16
combinaisons `(theta, r)`.

**Contrôle négatif** : `always_a` — non-dominante en action ; doit dégrader le
welfare par rapport à `always_a_prime` (cf §1 §1.1).

**Grades de pont** : chaque claim du carnet porte un grade A/C/T/B (Asserted / Calculated / Tested / Bridged), déclaré dans le § 8. Aucun passage C→B sans expérience préalable documentée.

In [2]:
// §0 — gel du protocole (asserts d'auto-coherence)
var PROFILS_VRAIS = Array.AsReadOnly(new (string Row, string Col)[]
{
    ("a", "a"), ("a", "a'"), ("a'", "a"), ("a'", "a'")
});
var POPULATIONS = Array.AsReadOnly(new[] { "uniform_random", "always_a_prime", "always_a", "byzantine" });
const string NULL_MODEL = "uniform_random";
const string NEGATIVE_CONTROL = "always_a";
const double H0_NULL_BAND = 0.1; // moyenne dans +/- 0.1 du nul => H0

// Les listes n'exposent aucune écriture ; valider le gel avant tout appel de stratégie.
bool ValiderGelV1()
{
    // Références citées : protocole et sortie de la cellule 4 du jumeau Python.
    var grainesCitees = new[] { 0, 1, 7, 42, 99 };
    VerifierEcart("§0", "nombre de graines", SEEDS.Count, grainesCitees.Length, TOL_EXACTE);
    for (int i = 0; i < grainesCitees.Length; i++)
        VerifierEcart("§0", Inv($"graine à l'indice {i}"), SEEDS[i], grainesCitees[i], TOL_EXACTE);
    VerifierEcart("§0", "taille par graine", N_PAIRS, 50, TOL_EXACTE);
    Exiger(PROFILS_VRAIS.SequenceEqual(new[] { ("a", "a"), ("a", "a'"), ("a'", "a"), ("a'", "a'") }),
        "Les quatre profils vrais et leur ordre doivent être conservés.");
    Exiger(POPULATIONS.SequenceEqual(new[] { "uniform_random", "always_a_prime", "always_a", "byzantine" }),
        "Les populations et leur ordre doivent être conservés.");
    Exiger(NULL_MODEL == "uniform_random" && POPULATIONS.Contains(NULL_MODEL), "Modèle nul du gel.");
    Exiger(NEGATIVE_CONTROL == "always_a" && POPULATIONS.Contains(NEGATIVE_CONTROL), "Contrôle négatif du gel.");
    VerifierEcart("§0", "bande H0", H0_NULL_BAND, 0.1, TOL_EXACTE);
    return true;
}

bool protocoleV1Gele = ValiderGelV1();
void ExigerGelV1() => Exiger(protocoleV1Gele, "Le gel v1 doit être vérifié avant tout appel de stratégie.");
ExigerGelV1();

Console.WriteLine("§0 protocole gele :");
Console.WriteLine($"  SEEDS = [{string.Join(", ", SEEDS.Select(s => s.ToString(CultureInfo.InvariantCulture)))}]");
Console.WriteLine(Inv($"  N_PAIRS = {N_PAIRS}"));
Console.WriteLine($"  POPULATIONS = [{string.Join(", ", POPULATIONS.Select(p => "'" + p + "'"))}]");
Console.WriteLine($"  NULL_MODEL = {NULL_MODEL}");
Console.WriteLine($"  NEGATIVE_CONTROL = {NEGATIVE_CONTROL}");
Console.WriteLine(Inv($"  H0 band = +/- {H0_NULL_BAND}"));

§0 protocole gele :


  SEEDS = [0, 1, 7, 42, 99]


  N_PAIRS = 50


  POPULATIONS = ['uniform_random', 'always_a_prime', 'always_a', 'byzantine']


  NULL_MODEL = uniform_random


  NEGATIVE_CONTROL = always_a


  H0 band = +/- 0.1


## 1. Le mécanisme Othman-Sandholm Proposition 6 — adaptateur minimal depuis 16d

Reproduction **à l'identique** de la construction du carnet owner 16d §4.6 :
tables d'utilités, carte du mécanisme, oracle de bien-être social. Les
**asserts de non-divergence** confrontent l'adaptateur aux chiffres publiés
par le owner (la même liste d'asserts vit au 16d §1).

In [3]:
// §1 — Mécanisme Othman-Sandholm Prop. 6 (ré-importé verbatim du 16e §4.6)
var u_row = new Dictionary<string, Dictionary<string, int>>(StringComparer.Ordinal)
{
    ["o1"] = new Dictionary<string, int> { ["a"] = 1, ["a'"] = 3 },
    ["o2"] = new Dictionary<string, int> { ["a"] = 4, ["a'"] = 5 },
    ["o3"] = new Dictionary<string, int> { ["a"] = 0, ["a'"] = 0 },
    ["o4"] = new Dictionary<string, int> { ["a"] = 3, ["a'"] = 0 }
};
var u_col = new Dictionary<string, Dictionary<string, int>>(StringComparer.Ordinal)
{
    ["o1"] = new Dictionary<string, int> { ["a"] = 1, ["a'"] = 4 },
    ["o2"] = new Dictionary<string, int> { ["a"] = 0, ["a'"] = 0 },
    ["o3"] = new Dictionary<string, int> { ["a"] = 3, ["a'"] = 6 },
    ["o4"] = new Dictionary<string, int> { ["a"] = 0, ["a'"] = 0 }
};
var outcomes = Array.AsReadOnly(new[] { "o1", "o2", "o3", "o4" });
var M_MAP = new Dictionary<(string Row, string Col), string>
{
    [("a'", "a'")] = "o1", [("a'", "a")] = "o2",
    [("a", "a'")] = "o3", [("a", "a")] = "o4"
};

string mecanisme(string r_row, string r_col) => M_MAP[(r_row, r_col)];
int sw_oracle(string outcome, string theta_row, string theta_col) => u_row[outcome][theta_row] + u_col[outcome][theta_col];
int payoff(string role, string outcome, string theta) => (role == "row" ? u_row : u_col)[outcome][theta];
int sw_optimal(string theta_row, string theta_col) => outcomes.Max(o => sw_oracle(o, theta_row, theta_col));

// Non-divergence vs owner 16e §4.6.3
// Références citées : les sept valeurs de SW des assertions de la cellule 6 du Python.
var temoinsSwCites = new[]
{
    (Row: "a'", Col: "a'", Cas: new[] { (Issue: "o1", Sw: 7) }),
    (Row: "a", Col: "a", Cas: new[] { (Issue: "o2", Sw: 4), (Issue: "o1", Sw: 2) }),
    (Row: "a", Col: "a'", Cas: new[] { (Issue: "o3", Sw: 6), (Issue: "o1", Sw: 5) }),
    (Row: "a'", Col: "a", Cas: new[] { (Issue: "o2", Sw: 5), (Issue: "o1", Sw: 4) })
};
int profilsControles = 0;
foreach (var (thetaRow, thetaCol, cas) in temoinsSwCites)
{
    foreach (var (issue, swCite) in cas)
        VerifierEcart("§1", $"SW({issue}, {thetaRow}, {thetaCol})", sw_oracle(issue, thetaRow, thetaCol), swCite, TOL_EXACTE);
    profilsControles++;
}
VerifierEcart("§1", "groupes d'assertions SW", profilsControles, 4, TOL_EXACTE);

// Dominance stricte de a' en action (8 cas)
int casDominance = 0;
foreach (string role in new[] { "row", "col" })
foreach (string t_vrai in new[] { "a", "a'" })
foreach (string r_adv in new[] { "a", "a'" })
{
    string o_a_prime = role == "row" ? mecanisme("a'", r_adv) : mecanisme(r_adv, "a'");
    string o_a = role == "row" ? mecanisme("a", r_adv) : mecanisme(r_adv, "a");
    int payoff_a = payoff(role, o_a, t_vrai);
    int payoff_aprime = payoff(role, o_a_prime, t_vrai);
    Exiger(payoff_aprime > payoff_a, Inv($"a' doit dominer a pour {role} type={t_vrai} adv={r_adv} : {payoff_aprime} <= {payoff_a}"));
    casDominance++;
}
VerifierEcart("§1", "cas de dominance stricte", casDominance, 8, TOL_EXACTE);

// Optima cités : sortie de la cellule 6 du Python, dans l'ordre des profils gelés.
var optimaCites = new[] { 4, 6, 5, 7 };
var optimaCalcules = PROFILS_VRAIS.ToDictionary(p => p, p => sw_optimal(p.Row, p.Col));
for (int i = 0; i < PROFILS_VRAIS.Count; i++)
{
    var p = PROFILS_VRAIS[i];
    VerifierEcart("§1", $"SW optimal ({p.Row}, {p.Col})", optimaCalcules[p], optimaCites[i], TOL_EXACTE);
}

// Respecter les guillemets de la sortie Python, notamment pour a'.
string AfficherType(string type) => type == "a'" ? "\"a'\"" : "'a'";
var texteOptima = string.Join(", ", PROFILS_VRAIS.Select(p =>
    Inv($"({AfficherType(p.Row)}, {AfficherType(p.Col)}): {optimaCalcules[p]}")));
Console.WriteLine(Inv($"§1 adaptateur non divergent : {profilsControles} asserts SW + {casDominance} asserts dominance stricte de a'."));
Console.WriteLine("SW optimal par profil vrai : {" + texteOptima + "}");

§1 adaptateur non divergent : 4 asserts SW + 8 asserts dominance stricte de a'.


SW optimal par profil vrai : {('a', 'a'): 4, ('a', "a'"): 6, ("a'", 'a'): 5, ("a'", "a'"): 7}


**Lecture chiffrée — §1.** Les 4 asserts SW confrontent l'oracle ici aux
chiffres publiés par le 16d §4.6.3. Les 8 asserts de dominance stricte (2 rôles
× 2 types × 2 rapports adverses) vérifient case par case que rapporter `a'`
domine strictement `a`, quel que soit le contexte — c'est le socle du
**contrôle négatif `always_a`** au §6 (qui doit perdre face à
`always_a_prime`). Le dictionnaire `SW optimal par profil vrai` sert de
référence au §4 : chaque ligne de trace portera son `sw_optimal` lu ici.

## 2. Définition d'une population — distribution sur (theta, r)

Une **population** = un triplet `(sample_theta, sample_r_row, sample_r_col)`
de trois fonctions qui, étant donné un seed et un index `i`, produisent
respectivement :

1. `theta = (theta_row, theta_col)` — le profil vrai de la paire `i`
2. `r_row` — le rapport d'action du joueur row
3. `r_col` — le rapport d'action du joueur col

Chaque politique (uniform_random, always_a_prime, always_a, byzantine)
conserve la même **distribution sur `theta`** (uniforme sur les 4 profils)
et se différencie par **sa règle d'action** (rapport déterministe ou tirage).

In [4]:
// §2 — Définition des populations
var ACTIONS = Array.AsReadOnly(new[] { "a", "a'" });

(string Row, string Col) theta_uniform(Func<int, int> choix)
{
    // Modele nul : theta uniforme sur les 4 profils.
    return PROFILS_VRAIS[choix(PROFILS_VRAIS.Count)];
}

(string Row, string Col) theta_uniform_stratified(Func<int, int> choix)
{
    // Profil de theta distribue comme un independant des marginales a et a'.
    return (ACTIONS[choix(ACTIONS.Count)], ACTIONS[choix(ACTIONS.Count)]);
}

// --- Politiques d'action ---
string r_uniform(Func<int, int> choix) => ACTIONS[choix(ACTIONS.Count)];
string r_always_aprime() => "a'";
string r_always_a() => "a";
string r_byzantine(string theta_self)
{
    // Byzantin : rapporte le complementaire de son type vrai.
    return theta_self == "a'" ? "a" : "a'";
}

// --- Politiques completes ---
((string Row, string Col) Theta, string ReportRow, string ReportCol) pop_uniform_random(Func<int, int> choix)
{
    // Population null model : theta uniforme + actions uniformes.
    var th = theta_uniform(choix);
    return (th, r_uniform(choix), r_uniform(choix));
}

((string Row, string Col) Theta, string ReportRow, string ReportCol) pop_always_a_prime_uniform(Func<int, int> choix)
{
    // Population dominante : theta uniforme (couverture) + toujours a'.
    var th = theta_uniform(choix);
    return (th, r_always_aprime(), r_always_aprime());
}

((string Row, string Col) Theta, string ReportRow, string ReportCol) pop_always_a_uniform(Func<int, int> choix)
{
    // Population controle negatif : theta uniforme + toujours a.
    var th = theta_uniform(choix);
    return (th, r_always_a(), r_always_a());
}

((string Row, string Col) Theta, string ReportRow, string ReportCol) pop_byzantine_uniform(Func<int, int> choix)
{
    // Population byzantine : theta uniforme + complement du type vrai.
    var th = theta_uniform(choix);
    return (th, r_byzantine(th.Row), r_byzantine(th.Col));
}

var POP_FNS = new System.Collections.ObjectModel.ReadOnlyDictionary<string, Func<Func<int, int>, ((string Row, string Col) Theta, string ReportRow, string ReportCol)>>(
    new Dictionary<string, Func<Func<int, int>, ((string Row, string Col) Theta, string ReportRow, string ReportCol)>>(StringComparer.Ordinal)
    {
        ["uniform_random"] = pop_uniform_random,
        ["always_a_prime"] = pop_always_a_prime_uniform,
        ["always_a"] = pop_always_a_uniform,
        ["byzantine"] = pop_byzantine_uniform
    });
Exiger(new HashSet<string>(POP_FNS.Keys, StringComparer.Ordinal).SetEquals(POPULATIONS),
    "POP_FNS doit couvrir POPULATIONS.");
// Ordre d'affichage explicite et référence citée de la cellule 9 du Python.
var populationsDefinies = POPULATIONS.Where(POP_FNS.ContainsKey).ToArray();
var textePopulations = "[" + string.Join(", ", populationsDefinies.Select(p => "'" + p + "'")) + "]";
Exiger(textePopulations == "['uniform_random', 'always_a_prime', 'always_a', 'byzantine']",
    "§2 : liste des populations différente de la sortie Python citée.");
Console.WriteLine($"§2 populations definies : {textePopulations}");

§2 populations definies : ['uniform_random', 'always_a_prime', 'always_a', 'byzantine']


**Lecture — §2.** Quatre populations sont définies par les fonctions
`POP_FNS`. La distribution `theta` est uniforme sur les 4 profils dans toutes
les populations : c'est la condition de **comparabilité** — ce qui varie entre
populations, c'est uniquement la règle d'action. Le contrôle négatif partage
la même distribution theta avec `always_a_prime` (couverture identique) : la
différence de welfare vient **purement** de l'action, pas du tirage de theta.
La population byzantine est la seule à connaître le type vrai (elle en
tire le rapport complémentaire) — c'est la sémantique du *Byzantine player*
du papier Othman-Sandholm, dont le 16d §4.6 testait déjà la dominance.

## 3. Driver multi-seed — exécution reproductible

Pour chaque `(population, seed)`, on tire `N_PAIRS` paires
`(theta, r_row, r_col)`, on applique le mécanisme pour produire l'outcome,
on lit le welfare via `sw_oracle`, et on accumule les résultats dans un
tableau indexé par `(pop, seed)`.

**Hypothèse** : le RNG du jumeau Python est `random.Random(seed)` ; le
rejeu C# reproduit ses choix d'indices pour les graines gelées. On consomme
`N_PAIRS × 3` choix pour `uniform_random` (theta, r_row, r_col), et
`N_PAIRS` pour chacune des autres populations (theta seulement). Les sorties
Python citées servent de références aux calculs du rejeu ; les indices sont
reconstitués ici, et non cités comme un relevé des tirages.

In [5]:
// §3 — Exécution multi-seed reproductible
// Reconstitution des indices de random.Random(seed).choice pour les graines
// positives ou nulles tenant dans un mot : init_by_array, MT19937, getrandbits
// et rejet jusqu'à un indice valide. Aucun tirage Python stocké n'est supposé.
class ChoixCPython
{
    const int N = 624;
    const int M = 397;
    readonly uint[] etat = new uint[N];
    int position = N;
    public int NombreDeChoix { get; private set; }

    public ChoixCPython(uint graine)
    {
        unchecked
        {
            etat[0] = 19650218U;
            for (int k = 1; k < N; k++)
                etat[k] = 1812433253U * (etat[k - 1] ^ (etat[k - 1] >> 30)) + (uint)k;

            // La clé d'initialisation de ces graines entières contient un seul mot.
            int i = 1;
            for (int k = 0; k < N; k++)
            {
                uint precedent = etat[i - 1];
                etat[i] = (etat[i] ^ ((precedent ^ (precedent >> 30)) * 1664525U)) + graine;
                if (++i == N)
                {
                    etat[0] = etat[N - 1];
                    i = 1;
                }
            }
            for (int k = 0; k < N - 1; k++)
            {
                uint precedent = etat[i - 1];
                etat[i] = (etat[i] ^ ((precedent ^ (precedent >> 30)) * 1566083941U)) - (uint)i;
                if (++i == N)
                {
                    etat[0] = etat[N - 1];
                    i = 1;
                }
            }
            etat[0] = 0x80000000U;
        }
    }

    uint MotSuivant()
    {
        if (position == N)
        {
            // Retorsion en place : après le bouclage, les mots sont déjà renouvelés.
            for (int i = 0; i < N; i++)
            {
                uint mot = (etat[i] & 0x80000000U) | (etat[(i + 1) % N] & 0x7fffffffU);
                etat[i] = etat[(i + M) % N] ^ (mot >> 1) ^ ((mot & 1U) == 0U ? 0U : 0x9908b0dfU);
            }
            position = 0;
        }
        uint y = etat[position++];
        y ^= y >> 11;
        y ^= (y << 7) & 0x9d2c5680U;
        y ^= (y << 15) & 0xefc60000U;
        return y ^ (y >> 18);
    }

    public int Suivant(int borne)
    {
        if (borne <= 0)
            throw new ArgumentOutOfRangeException(nameof(borne));
        // CPython demande bit_length(borne), y compris pour une puissance de deux.
        int bits = 0;
        for (int n = borne; n != 0; n >>= 1)
            bits++;
        uint indice;
        do { indice = MotSuivant() >> (32 - bits); }
        while (indice >= (uint)borne);
        NombreDeChoix++;
        return (int)indice;
    }
}

List<(int I, string ThetaRow, string ThetaCol, string RRow, string RCol, string Outcome, int Sw, int SwOptimal, int GapToOpt)> run_population(string pop_name, int seed)
{
    // Execute N_PAIRS paires independantes selon la politique `pop_name`.
    ExigerGelV1();
    var rng = new ChoixCPython(checked((uint)seed)); // RNG local : determinisme par seed
    var fn = POP_FNS[pop_name];
    var rows = new List<(int I, string ThetaRow, string ThetaCol, string RRow, string RCol, string Outcome, int Sw, int SwOptimal, int GapToOpt)>(N_PAIRS);
    for (int i = 0; i < N_PAIRS; i++)
    {
        var (th, r_row, r_col) = fn(rng.Suivant);
        string outcome = mecanisme(r_row, r_col);
        int sw = sw_oracle(outcome, th.Row, th.Col);
        int sw_opt = sw_optimal(th.Row, th.Col);
        Exiger(sw <= sw_opt, Inv($"§3 : welfare supérieur à l'optimum pour {pop_name}, seed {seed}, paire {i}."));
        rows.Add((i, th.Row, th.Col, r_row, r_col, outcome, sw, sw_opt, sw_opt - sw));
    }
    int choixAttendus = N_PAIRS * (pop_name == NULL_MODEL ? 3 : 1);
    VerifierEcart("§3", Inv($"choix d'indices {pop_name}, seed {seed}"), rng.NombreDeChoix, choixAttendus, TOL_EXACTE);
    VerifierEcart("§3", Inv($"paires {pop_name}, seed {seed}"), rows.Count, N_PAIRS, TOL_EXACTE);
    return rows;
}

// Une séquence explicite : population dans l'ordre du gel, puis graine, puis indice.
var lotsV1 = (from pop in POPULATIONS from seed in SEEDS select (Pop: pop, Seed: seed)).ToArray();
var results = lotsV1.ToDictionary(lot => lot, lot => run_population(lot.Pop, lot.Seed));
int total = results.Values.Sum(rows => rows.Count);
// Références citées : sortie de la cellule 12 du jumeau Python.
VerifierEcart("§3", "couples (pop, seed)", results.Count, 20, TOL_EXACTE);
VerifierEcart("§3", "paires totales", total, 1000, TOL_EXACTE);
Console.WriteLine(Inv($"§3 execution : {results.Count} couples (pop, seed), {N_PAIRS} paires chacun."));
Console.WriteLine(Inv($"  Total paires jouees : {total}"));

§3 execution : 20 couples (pop, seed), 50 paires chacun.


  Total paires jouees : 1000


**Lecture — §3.** 4 populations × 5 seeds × 50 paires = **1000 paires
jouées** au total. Le RNG est local à chaque couple `(pop, seed)` ; à seed
égal, les deux politiques constantes consomment la même séquence de profils.
Le triplet `(theta, r_row, r_col)`
de chaque paire est tracé en clair dans `rows` ; on pourra rejouer
l'agrégation au §4 sans toucher au RNG, ce qui protège la reproductibilité.

## 4. Agrégation par (population, seed) — distribution du welfare

Pour chaque couple `(pop, seed)`, on agrège la distribution du welfare :
moyenne, médiane, écart-type, comptes par outcome, comptes par profil vrai,
gap moyen à l'optimum. C'est la base des verdicts H0/H1 (au §7).

In [6]:
// §4 — Agrégation par (pop, seed)
// agg[(pop, seed)] = {mean, median, stdev, outcome_counts, theta_counts, gap_mean}
const int seedSyntheseV1 = 0;
// Les écarts-types sont comparés à des références citées à deux décimales.
double tolStdevPublieV1 = TolerancePubliee(2);
var agg = new Dictionary<(string Pop, int Seed), (double Mean, double Median, double Stdev, int Min, int Max, double GapMean, int N, Dictionary<string, int> OutcomeCounts, Dictionary<(string Row, string Col), int> ThetaCounts)>();
foreach (var lot in lotsV1)
{
    var rows = results[lot];
    var sws = rows.Select(r => r.Sw).OrderBy(sw => sw).ToArray();
    int n = sws.Length;
    double mean = sws.Average();
    double median = n % 2 == 0 ? (sws[n / 2 - 1] + sws[n / 2]) / 2.0 : sws[n / 2];
    // statistics.stdev utilise la variance d'échantillon, de dénominateur n - 1.
    double sommeCarres = sws.Sum(sw => (sw - mean) * (sw - mean));
    double stdev = n > 1 ? Math.Sqrt(sommeCarres / (n - 1)) : 0.0;
    var outcomeCounts = rows.GroupBy(r => r.Outcome).ToDictionary(g => g.Key, g => g.Count(), StringComparer.Ordinal);
    var thetaCounts = rows.GroupBy(r => (Row: r.ThetaRow, Col: r.ThetaCol)).ToDictionary(g => g.Key, g => g.Count());
    Exiger(outcomeCounts.Values.Sum() == n && thetaCounts.Values.Sum() == n,
        Inv($"§4 : comptage incomplet pour {lot.Pop}, seed {lot.Seed}."));
    agg.Add(lot, (mean, median, stdev, sws[0], sws[n - 1], rows.Average(r => r.GapToOpt), n, outcomeCounts, thetaCounts));
}

// Références citées : sortie de la cellule 15 du jumeau Python, seed 0.
// Outcomes dans l'ordre o1, o2, o3, o4 ; zéro pour une issue absente de la sortie.
var syntheseV1Citee = new Dictionary<string, (double Mean, double Median, double Stdev, double GapMean, int[] Effectifs)>(StringComparer.Ordinal)
{
    ["uniform_random"] = (3.40, 3.0, 1.80, 1.94, new[] { 12, 10, 13, 15 }),
    ["always_a_prime"] = (4.46, 4.0, 1.85, 1.00, new[] { 50, 0, 0, 0 }),
    ["always_a"] = (1.38, 0.0, 1.51, 4.08, new[] { 0, 0, 0, 50 }),
    ["byzantine"] = (2.16, 2.0, 1.46, 3.30, new[] { 13, 10, 14, 13 })
};
VerifierEcart("§4", "graine de la synthèse", seedSyntheseV1, 0, TOL_EXACTE);
foreach (string pop in POPULATIONS)
{
    var a = agg[(pop, seedSyntheseV1)];
    var reference = syntheseV1Citee[pop];
    // Pour 50 valeurs entières, les moyennes ont au plus deux décimales et la médiane une seule.
    VerifierEcart("§4", $"{pop} moyenne", a.Mean, reference.Mean, TOL_FLOTTANTE);
    VerifierEcart("§4", $"{pop} médiane", a.Median, reference.Median, TOL_EXACTE);
    VerifierEcart("§4", $"{pop} écart-type", a.Stdev, reference.Stdev, tolStdevPublieV1);
    Exiger(Inv($"{a.Stdev:F2}") == Inv($"{reference.Stdev:F2}"), $"§4 : arrondi de l'écart-type différent pour {pop}.");
    VerifierEcart("§4", $"{pop} écart moyen à l'optimum", a.GapMean, reference.GapMean, TOL_FLOTTANTE);
    Exiger(a.OutcomeCounts.Keys.All(outcomes.Contains), $"§4 : issue inattendue pour {pop}.");
    for (int i = 0; i < outcomes.Count; i++)
    {
        string issue = outcomes[i];
        int effectif = a.OutcomeCounts.TryGetValue(issue, out int compte) ? compte : 0;
        VerifierEcart("§4", $"{pop} effectif {issue}", effectif, reference.Effectifs[i], TOL_EXACTE);
    }
}

// Affiche synthese pour une seed representative (seed=0)
Console.WriteLine(Inv($"§4 — Distribution du welfare (seed={seedSyntheseV1}) :"));
Console.WriteLine(Inv($"{"pop",-18} {"mean",6} {"median",7} {"stdev",6} {"gap_mean",9}  outcomes"));
foreach (string pop in POPULATIONS)
{
    var a = agg[(pop, seedSyntheseV1)];
    string texteOutcomes = string.Join(" ", a.OutcomeCounts.OrderBy(kv => kv.Key, StringComparer.Ordinal)
        .Select(kv => Inv($"{kv.Key}:{kv.Value}")));
    Console.WriteLine(Inv($"{pop,-18} {a.Mean,6:F2} {a.Median,7:F1} {a.Stdev,6:F2} {a.GapMean,9:F2}  {texteOutcomes}"));
}

§4 — Distribution du welfare (seed=0) :


pop                  mean  median  stdev  gap_mean  outcomes


uniform_random       3.40     3.0   1.80      1.94  o1:12 o2:10 o3:13 o4:15


always_a_prime       4.46     4.0   1.85      1.00  o1:50


always_a             1.38     0.0   1.51      4.08  o4:50


byzantine            2.16     2.0   1.46      3.30  o1:13 o2:10 o3:14 o4:13


**Lecture — §4 (seed 0).** Les profils vrais sont tirés uniformément
sur les 4 profils par construction (§2). Dans cet échantillon, chaque population rencontre
les 4 profils. La distribution d'outcomes reflète la politique d'action :

- `uniform_random` : les quatre couples de rapports sont équiprobables par construction.
  Dans l'échantillon, on compte 12 paires pour `o1`, 10 pour `o2`, 13 pour `o3` et 15 pour `o4`.
- `always_a_prime` : un seul outcome `o1` (puisque `r_row = r_col = a'`).
- `always_a` : un seul outcome `o4` (puisque `r_row = r_col = a`).
- `byzantine` : chaque joueur rapporte le complémentaire, donc
  `(r_row, r_col) = (complement(theta_row), complement(theta_col))` — 4 cases
  visitées selon le profil vrai.

Le `gap_mean` (moyenne de `sw_optimal - sw_oracle`) mesure l'écart à l'optimum,
et l'ordre mesuré au seed 0 est `always_a` (4.08) > `byzantine` (3.30) >
`uniform_random` (1.94) > `always_a_prime` (1.00).

`always_a_prime` n'est **pas** à `gap=0` : un rapport constant `a'` n'est
exactement optimal que sur le profil vrai `(a',a')` — là `sw_optimal = 7` et
le mécanisme rend `o1` (welfare 7), donc gap 0. Sur les trois autres profils,
le même rapport constant laisse un écart de 2 sur `(a,a)`, 1 sur `(a,a')` et
1 sur `(a',a)` ; la moyenne sur les quatre profils équiprobables vaut donc
**1.00**, et c'est exactement la valeur mesurée. La dominance de `a'` (§1) est
une dominance **en action** — `a'` rapporte plus que `a` dans chaque contexte —
pas une optimalité de bien-être sur tous les profils. C'est `always_a`, le
rapport perdant, qui porte le gap le plus large : 4.00 en moyenne sur les
quatre profils (4.08 mesuré).

## 5. Modèle nul — uniform_random

Le modèle nul `uniform_random` est défini au §0. Il joue le rôle de
**baseline inerte** : sous H0, le welfare moyen des autres populations reste
dans la bande fixée autour du nul. La
**comparaison au nul** se fait seed par seed : pour chaque population
non-nulle, on regarde `mean(pop) - mean(null)` (différence de moyenne par
seed). Si toutes les populations sont dans la bande ±0.1 du nul, on
accepte H0 ; sinon, on rejette H0 au profit de H1.

La bande est une règle de décision pré-enregistrée, pas un test de significativité statistique.

In [7]:
// §5 — Comparaison au modèle nul (uniform_random)
// Références citées : sortie de la cellule 18 du jumeau Python, dans l'ordre des graines gelées.
var moyennesNullesCitees = new[] { 3.40, 4.24, 3.50, 4.24, 4.08 };
const double moyenneGlobaleNulleCitee = 3.8920;
var comparaisonsNullesCitees = new Dictionary<string, (double[] ParSeed, double MoyennePubliee, double Maximum, bool DansBande)>(StringComparer.Ordinal)
{
    ["always_a_prime"] = (new[] { +1.06, +0.42, +0.60, -0.32, +0.68 }, +0.49, 1.06, false),
    ["always_a"] = (new[] { -2.02, -2.80, -1.58, -2.32, -2.34 }, -2.21, 2.80, false),
    ["byzantine"] = (new[] { -1.24, -2.34, -1.22, -1.78, -1.78 }, -1.67, 2.34, false)
};
// Un écart par graine est un multiple de 1/50 ; seule la moyenne des écarts est citée arrondie au centième.
double tolMoyenneEcartV1 = TolerancePubliee(2);
string Signe2(double x) => Inv($"{x:+0.00;-0.00;+0.00}");
var nonNullesV1 = POPULATIONS.Where(pop => pop != NULL_MODEL).ToArray();
var null_means = SEEDS.ToDictionary(s => s, s => agg[(NULL_MODEL, s)].Mean);
double moyenneGlobaleNulle = SEEDS.Average(s => null_means[s]);
VerifierEcart("§5", "moyenne globale du nul", moyenneGlobaleNulle, moyenneGlobaleNulleCitee, TOL_FLOTTANTE);
for (int i = 0; i < SEEDS.Count; i++)
{
    int s = SEEDS[i];
    VerifierEcart("§5", Inv($"moyenne du nul, seed {s}"), null_means[s], moyennesNullesCitees[i], TOL_FLOTTANTE);
}

// diffs[(pop, seed)] = mean(pop, seed) - mean(null, seed)
var diffs = new Dictionary<(string Pop, int Seed), double>();
foreach (string pop in nonNullesV1)
for (int i = 0; i < SEEDS.Count; i++)
{
    int s = SEEDS[i];
    double d = agg[(pop, s)].Mean - null_means[s];
    VerifierEcart("§5", Inv($"écart au nul {pop}, seed {s}"), d, comparaisonsNullesCitees[pop].ParSeed[i], TOL_FLOTTANTE);
    diffs.Add((pop, s), d);
}

// Verdict preliminaire par population
var verdicts_null = new Dictionary<string, (double AvgDiff, double MaxAbsDiff, bool InBand)>(StringComparer.Ordinal);
foreach (string pop in nonNullesV1)
{
    var per_seed_diff = SEEDS.Select(s => diffs[(pop, s)]).ToArray();
    double avg_diff = per_seed_diff.Average();
    double max_abs = per_seed_diff.Max(d => Math.Abs(d));
    bool in_band = max_abs <= H0_NULL_BAND;
    var reference = comparaisonsNullesCitees[pop];
    VerifierEcart("§5", $"{pop} écart moyen au nul", avg_diff, reference.MoyennePubliee, tolMoyenneEcartV1);
    Exiger(Signe2(avg_diff) == Signe2(reference.MoyennePubliee), $"§5 : arrondi de l'écart moyen différent pour {pop}.");
    VerifierEcart("§5", $"{pop} écart absolu maximal", max_abs, reference.Maximum, TOL_FLOTTANTE);
    Exiger(in_band == reference.DansBande, $"§5 : verdict préliminaire différent pour {pop}.");
    verdicts_null.Add(pop, (avg_diff, max_abs, in_band));
}

string texteNul = string.Join(", ", SEEDS.Select(s => Inv($"{s}: {null_means[s]:0.0#}")));
Console.WriteLine($"§5 — Moyenne du modele nul ({NULL_MODEL}) par seed : {{" + texteNul + "}");
Console.WriteLine(Inv($"  Moyenne globale du nul : {moyenneGlobaleNulle:F4}"));
Console.WriteLine();
Console.WriteLine("§5 — Differences au nul par population × seed :");
Console.WriteLine(Inv($"{"pop",-18} ") + string.Join(" ", SEEDS.Select(s => Inv($"s={s,2}"))) + "  mean_diff");
foreach (string pop in nonNullesV1)
{
    string ligne = string.Join(" ", SEEDS.Select(s => Signe2(diffs[(pop, s)]).PadLeft(5)));
    Console.WriteLine(Inv($"{pop,-18} {ligne}  {Signe2(verdicts_null[pop].AvgDiff),5}"));
}
Console.WriteLine();
Console.WriteLine("§5 — Verdict preliminaire par population :");
foreach (string pop in nonNullesV1)
{
    var v = verdicts_null[pop];
    string label = v.InBand ? "H0 (dans la bande)" : "H1 (hors bande)";
    Console.WriteLine(Inv($"  {pop,-18} avg_diff={Signe2(v.AvgDiff),5}  max|diff|={v.MaxAbsDiff:F2}  {label}"));
}

§5 — Moyenne du modele nul (uniform_random) par seed : {0: 3.4, 1: 4.24, 7: 3.5, 42: 4.24, 99: 4.08}


  Moyenne globale du nul : 3.8920


§5 — Differences au nul par population × seed :


pop                s= 0 s= 1 s= 7 s=42 s=99  mean_diff


always_a_prime     +1.06 +0.42 +0.60 -0.32 +0.68  +0.49


always_a           -2.02 -2.80 -1.58 -2.32 -2.34  -2.21


byzantine          -1.24 -2.34 -1.22 -1.78 -1.78  -1.67


§5 — Verdict preliminaire par population :


  always_a_prime     avg_diff=+0.49  max|diff|=1.06  H1 (hors bande)


  always_a           avg_diff=-2.21  max|diff|=2.80  H1 (hors bande)


  byzantine          avg_diff=-1.67  max|diff|=2.34  H1 (hors bande)


**Lecture — §5.** Le modèle nul sert de baseline inerte. Une population
qui sort de la bande ±0.1 rejette H0 ; une population qui y reste l'accepte.
La bande est volontairement **étroite** (±0.1 sur une échelle 0-7) : on cherche des écarts
**structurels** (50 paires × 5 seeds = 250 mesures par population). Comparer
à seed égal n'apparie pas les profils au nul : `uniform_random` consomme
aussi des tirages pour ses rapports.

À ce stade du carnet (avant §7), `always_a_prime` montre un écart moyen
positif (+0.49), `always_a` un écart moyen négatif (−2.21), et
`byzantine` un écart moyen intermédiaire (−1.67). Le verdict
**définitif** H0/H1 se calcule au §7 sur la base de ces écarts.

## 6. Contrôle négatif — `always_a` doit dégrader le welfare

Le contrôle négatif `always_a` est défini au §0. Rapporter `a` est
strictement dominé par `a'` en utilité individuelle (cf §1 asserts de dominance).
Le contrôle porte sur un résultat distinct : **la population `always_a` doit
produire un welfare moyen strictement inférieur à `always_a_prime`** sur
**chacun des seeds du protocole**. Cela n'implique pas un avantage social
sur chaque profil vrai.

C'est le test négatif : s'il échoue, on vérifie l'oracle, les tirages
et l'application des deux politiques. On documente l'écart et on
rappelle la borne théorique `sw_optimal("a'", "a'") = 7` (optimal de
toute la matrice).

In [8]:
// §6 — Contrôle négatif : always_a < always_a_prime (oracle-dominance)
// Références citées : sortie de la cellule 21 et lecture des cellules 16 et 22 du jumeau Python.
var controleNegatifCite = new (double Prime, double A, double Diff)[]
{
    (4.460, 1.380, +3.080), (4.660, 1.440, +3.220), (4.100, 1.920, +2.180),
    (3.920, 1.920, +2.000), (4.760, 1.740, +3.020)
};
var ecartsProfilsCites = new[] { -1, 2, 4, 7 };
var gapsPrimeCites = new[] { 2, 1, 1, 0 };
int sw_ap = sw_optimal("a'", "a'");
int sw_aa = sw_optimal("a", "a");
VerifierEcart("§6", "oracle (a,a)", sw_aa, 4, TOL_EXACTE);
VerifierEcart("§6", "oracle (a',a')", sw_ap, 7, TOL_EXACTE);
string issuePrime = mecanisme("a'", "a'");
string issueA = mecanisme("a", "a");
Exiger(issuePrime == "o1" && issueA == "o4", "§6 : issues des politiques constantes.");
VerifierEcart("§6", "SW de o1 sur (a,a)", sw_oracle(issuePrime, "a", "a"), 2, TOL_EXACTE);
VerifierEcart("§6", "SW de o4 sur (a,a)", sw_oracle(issueA, "a", "a"), 3, TOL_EXACTE);
Exiger(outcomes.Where(o => sw_oracle(o, "a", "a") == sw_aa).SequenceEqual(new[] { "o2" }),
    "§6 : o2 doit être l'unique optimum sur (a,a).");

// La moyenne sur les quatre profils équiprobables est distincte de la moyenne empirique.
var ecartsControleParProfil = new Dictionary<(string Row, string Col), int>();
var gapsPrimeParProfil = new List<int>();
var gapsAParProfil = new List<int>();
for (int i = 0; i < PROFILS_VRAIS.Count; i++)
{
    var p = PROFILS_VRAIS[i];
    int swPrime = sw_oracle(issuePrime, p.Row, p.Col);
    int swA = sw_oracle(issueA, p.Row, p.Col);
    int optimum = sw_optimal(p.Row, p.Col);
    int d = swPrime - swA;
    int gapPrime = optimum - swPrime;
    VerifierEcart("§6", $"écart o1 - o4 pour {p}", d, ecartsProfilsCites[i], TOL_EXACTE);
    VerifierEcart("§6", $"gap du rapport constant a' pour {p}", gapPrime, gapsPrimeCites[i], TOL_EXACTE);
    ecartsControleParProfil.Add(p, d);
    gapsPrimeParProfil.Add(gapPrime);
    gapsAParProfil.Add(optimum - swA);
}
VerifierEcart("§6", "écart théorique sur les quatre profils", ecartsControleParProfil.Values.Average(), 3.00, TOL_EXACTE);
VerifierEcart("§6", "gap théorique du rapport constant a'", gapsPrimeParProfil.Average(), 1.00, TOL_EXACTE);
VerifierEcart("§6", "gap théorique du rapport constant a", gapsAParProfil.Average(), 4.00, TOL_EXACTE);

// Apparier les profils à chaque indice avant de vérifier l'écart des deux populations.
var all_diffs = new List<double>();
for (int i = 0; i < SEEDS.Count; i++)
{
    int s = SEEDS[i];
    var lignesPrime = results[("always_a_prime", s)];
    var lignesA = results[(NEGATIVE_CONTROL, s)];
    Exiger(lignesPrime.Select(r => (r.I, r.ThetaRow, r.ThetaCol))
        .SequenceEqual(lignesA.Select(r => (r.I, r.ThetaRow, r.ThetaCol))),
        Inv($"§6 : profils non appariés, seed {s}."));
    double a_prime = agg[("always_a_prime", s)].Mean;
    double a = agg[(NEGATIVE_CONTROL, s)].Mean;
    double d = a_prime - a;
    var reference = controleNegatifCite[i];
    VerifierEcart("§6", Inv($"moyenne always_a_prime, seed {s}"), a_prime, reference.Prime, TOL_FLOTTANTE);
    VerifierEcart("§6", Inv($"moyenne always_a, seed {s}"), a, reference.A, TOL_FLOTTANTE);
    VerifierEcart("§6", Inv($"écart des moyennes, seed {s}"), d, reference.Diff, TOL_FLOTTANTE);
    double ecartApparie = lignesPrime.Average(r => ecartsControleParProfil[(r.ThetaRow, r.ThetaCol)]);
    VerifierEcart("§6", Inv($"écart des moyennes vs moyenne appariée, seed {s}"), d, ecartApparie, TOL_FLOTTANTE);
    all_diffs.Add(d);
}
double diffMoyenneControle = all_diffs.Average();
double minAbsControle = all_diffs.Min(d => Math.Abs(d));
int positive_count = all_diffs.Count(d => d > 0.0);
double pourcentagePositif = 100.0 * positive_count / SEEDS.Count;
VerifierEcart("§6", "écart moyen du contrôle", diffMoyenneControle, 2.700, TOL_FLOTTANTE);
VerifierEcart("§6", "écart absolu minimal", minAbsControle, 2.000, TOL_FLOTTANTE);
VerifierEcart("§6", "écart absolu maximal", all_diffs.Max(d => Math.Abs(d)), 3.220, TOL_FLOTTANTE);
VerifierEcart("§6", "graines du contrôle", SEEDS.Count, 5, TOL_EXACTE);
VerifierEcart("§6", "graines avec écart positif", positive_count, 5, TOL_EXACTE);
Exiger(positive_count == SEEDS.Count, "always_a_prime doit dominer always_a sur CHAQUE seed.");
VerifierEcart("§6", "pourcentage avec écart positif", pourcentagePositif, 100.0, TOL_EXACTE);

Console.WriteLine("§6 — Controle negatif : dominance stricte de a' face a l'oracle.");
Console.WriteLine(Inv($"  (oracle : sw_optimal(a,a) = {sw_aa} | sw_optimal(a_prime,a_prime) = {sw_ap} )"));
Console.WriteLine();
Console.WriteLine("§6 — Welfare par seed : always_a_prime vs always_a");
Console.WriteLine(Inv($"{"seed",-6} {"a_prime",9} {"a",9} {"diff",7}"));
for (int i = 0; i < SEEDS.Count; i++)
{
    int s = SEEDS[i];
    double a_prime = agg[("always_a_prime", s)].Mean;
    double a = agg[(NEGATIVE_CONTROL, s)].Mean;
    Console.WriteLine(Inv($"{s,-6} {a_prime,9:F3} {a,9:F3} {all_diffs[i],7:+0.000;-0.000;+0.000}"));
}
Console.WriteLine();
Console.WriteLine(Inv($"§6 — Diff moyenne : {diffMoyenneControle:+0.000;-0.000;+0.000}, min |diff| : {minAbsControle:F3}"));
Console.WriteLine(Inv($"§6 — Seeds avec always_a_prime > always_a : {positive_count}/{SEEDS.Count}"));
Console.WriteLine(Inv($"§6 — Controle negatif OK : dominance stricte de a' verifiee sur {pourcentagePositif:F0}% des seeds."));

§6 — Controle negatif : dominance stricte de a' face a l'oracle.


  (oracle : sw_optimal(a,a) = 4 | sw_optimal(a_prime,a_prime) = 7 )


§6 — Welfare par seed : always_a_prime vs always_a


seed     a_prime         a    diff


0          4.460     1.380  +3.080


1          4.660     1.440  +3.220


... (2 outputs left out)

99         4.760     1.740  +3.020


§6 — Diff moyenne : +2.700, min |diff| : 2.000


§6 — Seeds avec always_a_prime > always_a : 5/5


§6 — Controle negatif OK : dominance stricte de a' verifiee sur 100% des seeds.


**Lecture — §6.** L'écart `always_a_prime - always_a` mesure la différence de
welfare entre le rapport dominant et le rapport dominé, population contre
population. Il est **positif sur les 5 seeds** (+3.080, +3.220, +2.180,
+2.000, +3.020), d'où une moyenne de **+2.700** et un minimum de `|écart| =
2.000` : c'est ce que l'assert du §6 exige, et il tient à 100 %.

Sa valeur **n'est pas constante** d'un seed à l'autre. Par profil vrai, la différence
`sw(o1) - sw(o4)` vaut `-1` sur `(a,a)`, `+2` sur `(a,a')`, `+4` sur `(a',a)`
et `+7` sur `(a',a')` : sur le profil `(a,a)`, le rapport constant `a'` est
socialement **moins** bon que `a` (`o1` rend 2, `o4` rend 3) — l'optimum y est
`o2`, qui rend 4. La moyenne sur les quatre profils équiprobables vaut donc
`3.00`, et les 50 paires par seed l'échantillonnent avec le bruit attendu
(`2.000` à `3.220` observés sur les 5 seeds).

C'est **le** test négatif du pilote : si le signe attendu n'est pas retrouvé
sur les graines gelées, le contrôle échoue.

## 7. Verdict H0/H1 par population

H0 : *toutes populations produisent le même welfare moyen que le nul, à ±0.1 près, sur tous les seeds.*
H1 : *au moins une population s'écarte du nul au-delà de cette bande.*

Le verdict se calcule seed par seed : pour chaque population non-nulle, on
regarde si sa moyenne par seed est dans la bande `±H0_NULL_BAND` autour du
nul. Une population qui sort de la bande **sur au moins un seed** rejette
H0 ; sinon elle accepte H0.

L'analyse par seed (et non par moyenne globale) protège contre les
compensations accidentelles entre seeds : une population peut être dans
la bande en moyenne globale mais s'écarter sur un seed spécifique.

In [9]:
// §7 — Verdict H0/H1 par population
// Références citées : verdicts et écarts moyens publiés dans la sortie de la cellule 24 du Python.
var verdictsV1Cites = new Dictionary<string, (string Verdict, double AvgDiff)>(StringComparer.Ordinal)
{
    ["uniform_random"] = ("BASELINE", 0.000),
    ["always_a_prime"] = ("H1", +0.488),
    ["always_a"] = ("H1", -2.212),
    ["byzantine"] = ("H1", -1.672)
};

// Appliquer la bande gelée aux écarts non arrondis, avant toute mise en forme.
var verdicts_h0 = new Dictionary<string, string>(StringComparer.Ordinal);
var ecartsMoyensV1 = new Dictionary<string, double>(StringComparer.Ordinal);
foreach (string pop in POPULATIONS)
{
    var parSeed = SEEDS.Select(s => agg[(pop, s)].Mean - agg[(NULL_MODEL, s)].Mean).ToArray();
    string verdict = pop == NULL_MODEL ? "BASELINE" :
        parSeed.Max(d => Math.Abs(d)) <= H0_NULL_BAND ? "H0" : "H1";
    double ecartMoyen = parSeed.Average();
    var reference = verdictsV1Cites[pop];
    Exiger(verdict == reference.Verdict, $"§7 : {pop}, verdict calculé {verdict}, attendu {reference.Verdict}.");
    if (pop != NULL_MODEL)
        Exiger((verdict == "H0") == verdicts_null[pop].InBand, $"§7 : décision incohérente avec le §5 pour {pop}.");
    // Avec 5 × 50 observations entières, l'écart moyen est un multiple de 1/250.
    VerifierEcart("§7", $"{pop} écart moyen au nul", ecartMoyen, reference.AvgDiff, TOL_FLOTTANTE);
    verdicts_h0.Add(pop, verdict);
    ecartsMoyensV1.Add(pop, ecartMoyen);
}

int h0_count = verdicts_h0.Values.Count(v => v == "H0");
int h1_count = verdicts_h0.Values.Count(v => v == "H1");
int baseline_count = verdicts_h0.Values.Count(v => v == "BASELINE");
VerifierEcart("§7", "effectif H0", h0_count, 0, TOL_EXACTE);
VerifierEcart("§7", "effectif H1", h1_count, 3, TOL_EXACTE);
VerifierEcart("§7", "effectif BASELINE", baseline_count, 1, TOL_EXACTE);

Console.WriteLine("§7 — Verdict final par population :");
foreach (string pop in POPULATIONS)
    Console.WriteLine(Inv($"  {pop,-18} verdict={verdicts_h0[pop]}  avg_diff_nul={ecartsMoyensV1[pop]:+0.000;-0.000;+0.000}"));
Console.WriteLine();
Console.WriteLine("§7 — Resume H0/H1 :");
Console.WriteLine(Inv($"  H0 (dans la bande) : {h0_count}"));
Console.WriteLine(Inv($"  H1 (hors bande)   : {h1_count}"));
Console.WriteLine(Inv($"  Baseline (nul)    : {baseline_count}"));

§7 — Verdict final par population :


  uniform_random     verdict=BASELINE  avg_diff_nul=+0.000


  always_a_prime     verdict=H1  avg_diff_nul=+0.488


  always_a           verdict=H1  avg_diff_nul=-2.212


  byzantine          verdict=H1  avg_diff_nul=-1.672


§7 — Resume H0/H1 :


  H0 (dans la bande) : 0


  H1 (hors bande)   : 3


  Baseline (nul)    : 1


**Lecture — §7.** Les populations scriptées comparées au nul sortent de la bande ±0.1
sur au moins un seed : `always_a_prime` montre un écart moyen positif
(+0.488), `always_a` un écart moyen négatif (−2.212, perte par rapport
au nul), `byzantine` un écart intermédiaire (−1.672, perturbation). H0 est rejeté
pour les trois populations scriptées au profit de H1. La dominance en
action (assertions du §1) est vérifiée sur la table ; l'écart de welfare est
mesuré sur la population. Le contrôle négatif `always_a` dégrade effectivement le
welfare moyen par rapport à `always_a_prime` sur les graines gelées. Les témoins sont :

1. **L'oracle M** (non-divergence avec le 16d §4.6 : les assertions du §1 contrôlent les valeurs de référence).
2. **Le cadre population** (les 50 paires/seed permettent de mesurer les écarts sur les graines gelées).
3. **Le protocole re-gelé** (H0/H1 *lisent* leur seuil dans le
   gel §0, pas dans les résultats).

L'extension LLM du §10 compare deux populations mixtes à leurs ancres
scriptées à partir des données citées du jumeau Python, sans appel LLM en C#.
Elle est hors périmètre de cette v1.

## Conclusion du protocole v1

Ce carnet **étend le pilote 16d** (Othman-Sandholm, Proposition 6) d'une
paire unique à des **populations** de 50 paires, avec 4 politiques
scriptées (uniform_random, always_a_prime, always_a, byzantine) et 5 seeds
parmi {0, 1, 7, 42, 99}.

**Résultats de verdicts** : H0 est rejeté pour `always_a_prime` (écart
moyen positif), `always_a` (écart moyen négatif,
contrôle négatif) et `byzantine` (écart intermédiaire). Sur ces graines, le
rapport constant dominant obtient un welfare moyen supérieur à celui du
rapport constant dominé.

**Ce qui a été respecté** :
- Protocole re-gelé en §0 (avant toute exécution de stratégie).
- ≥ 2 populations hétérogènes (4 scriptées v1 + 2 mixtes LLM × scripts v2).
- Multi-seed ≥ 4 (5 retenus sur la liste standard).
- Modèle nul et contrôle négatif explicitement nommés et rapportés.
- Verdicts H0/H1 par population.

**Extension LLM (v2, §10)** : deux populations mixtes où l'agent
`row` est un LLM (condition `llm-regles-visibles` du 16d) et l'agent `col` reste
scripté, comparées à leurs ancres pures par **appariement exact des thetas**
(même séquence par seed, vérifiée par assertion). Dans les résultats cités du
jumeau Python, glm-5.2 rapporte l'action dominante à un taux de 1.00 dans les
 deux populations. Verdicts cités : `mix-dominant` **H1**
(diff +0.000 sur les 5 seeds — remplacer un côté scripté par un LLM qui suit
la règle ne coûte rien), `mix-byzantin` **H1** (diff moyenne +1.82 à deux décimales,
tous les écarts s'affichent à +0.92 ou plus — le rapport dominant amortit le coût
byzantin sans l'éliminer). La famille qwen3.8-max est exclue (429 persistants),
comptée à part — les deux issues étaient pré-enregistrées comme publiables.

## 8. Grades de pont A/C/T/B

Chaque claim mesurable du carnet porte un grade explicite, qui **trace la chaîne de preuve** entre l’énoncé et son témoin. La convention suit quatre grades :

- **A — Asserted** : constante ou énoncé garanti par une assertion C# au § 0. Lecture : le code refuse de démarrer si la valeur change.
- **C — Calculated** : valeur calculée par le carnet à partir des traces (`rows`). Lecture : la valeur est reproductible pour un seed donné avec les mêmes entrées.
- **T — Tested** : valeur confrontée à un témoin (les chiffres cités du pilote 16d §4.6.3 dans le § 1, l’appariement des profils dans le § 6). Lecture : la cellule échoue si la réplique dévie au-delà de la tolérance déclarée.
- **B — Bridged** : conclusion tirée d’une combinaison C + un seuil (H0/H1 ±0.1 dans le § 7). Lecture : le verdict est reproductible via le même calcul, le seuil est explicite, mais le pont n’est pas un théorème — c’est une décision de référentiel.

| Section | Claim | Grade | Témoin |
|---|---|---|---|
| § 0 | SEEDS = [0, 1, 7, 42, 99], N_PAIRS = 50, POPULATIONS ≥ 2 (4 effectives) | **A** | `ValiderGelV1` au § 0 |
| § 0 | H0 band ± 0.1, NULL_MODEL = uniform_random, NEGATIVE_CONTROL = always_a | **A** | Constantes et assertions du gel § 0 |
| § 1 | 4 groupes d'assertions SW (oracles) coïncident avec 16d §4.6.3 | **T** | Valeurs citées et comparaison exécutable au § 1 |
| § 1 | 8 assertions de dominance stricte (a’ > a, 2 rôles × 2 types × 2 rapports) | **T** | Comparaison des utilités par rôle, type et rapport adverse |
| § 2 | 4 populations hétérogènes (uniform_random, always_a_prime, always_a, byzantine) | **A** | `POP_FNS` + uniformité theta sur 4 profils |
| § 3 | 4 populations × 5 seeds × 50 paires = 1000 paires totales | **C** | Driver multi-seed et comptage des traces |
| § 4 | Distribution de welfare par (pop, seed) : moyenne, médiane, écart-type | **C** | Agrégation depuis `rows` |
| § 5 | Modèle nul : moyenne de welfare de `uniform_random` par seed | **C** | Moyenne par seed, base de comparaison H0 |
| § 6 | Contrôle négatif : `always_a` < `always_a_prime` en moyenne sur chaque seed gelé | **T** | Appariement des profils et assertions du § 6 |
| § 7 | H0 rejeté pour 3 populations scriptées (always_a_prime, always_a, byzantine), H1 retenu | **B** | C : moyenne par seed + A : H0 band ± 0.1 |
| § 10.0 | PREREG_V2 : 2 populations mixtes, 5 seeds, budget 240/320, hypothèses H0/H1 pré-enregistrées | **A** | Assertions du gel v2 avant le rejeu |
| § 10.1 | Validation de l'action et politique de reprise | **T** | Auto-tests du parseur et témoins hors ligne |
| § 10.2 | 240 paires dans le rejeu ; appariement exact des thetas sur les 5 seeds | **C/T** | Comptage et assertion d'égalité des séquences |
| § 10.2 | 480 tentatives HTTP, 120 réponses valides et 120 erreurs réseau dans l'expérience Python | **Cité** | Sortie historique du jumeau Python ; aucune tentative HTTP en C# |
| § 10.3 | Taux a′ LLM = 1.00 (glm, 2 populations) ; verdicts H1 / H1 (glm), EXCLU ×2 (qwen) | **B** | C : agrégats à partir des données citées + A : bande ±0.1 + T : appariement |
| Conclusion | Généralisation du pilote 16d aux populations étudiées | **B** | B : verdicts + T : oracle § 1 + C : 1000 paires § 3 |

**Règle C→B** : aucun passage C→B n’est autorisé sans expérience documentée. Les passages C→B du carnet (verdict § 7, verdict § 10.3, Conclusion) sont **précédés** d’un témoin T (oracle § 1, contrôle § 6, appariement § 10.2). Les témoins documentent le protocole re-gelé, les populations hétérogènes, les cinq graines et les grades de chaque affirmation.

**Limites méthodologiques des grades B** : un pont B n’est pas une preuve au sens fort — c’est un **contrat de référentiel** (le seuil H0 band ± 0.1, les tables du mécanisme, le témoin T amont) qui rend la conclusion reproductible **tant que le référentiel tient**. Trois conséquences opérationnelles :

- **Sensibilité au seuil** : déplacer la bande H0 peut faire basculer une conclusion. Le § 7 reste valide **pour ±0.1**, pas en général. Toute relecture du carnet doit citer le seuil exact.
- **Dépendance au témoin T** : un pont B héberge la défaillance de son témoin. Si les tables du mécanisme changent, il faut refaire les contrôles et les calculs — c’est pourquoi la table de § 8 trace le témoin de chaque B, pas seulement le calcul.
- **Non-portée hors graines** : le verdict est **établi sur les 5 graines [0, 1, 7, 42, 99]**, pas sur l’ensemble infini. Une généralisation au-delà demande une nouvelle expérience.

Ces trois limites sont **assumées** par le grade B : le carnet ne franchit pas le pont, il documente les conditions du pont.

## §9. Écarts d'exécution vs gel (consignés, conformément au §0)

Le rejeu v1 conserve les paramètres du gel §0 :

- **SEEDS** : {0, 1, 7, 42, 99}.
- **N_PAIRS = 50** — vérifié par assertion au §0.
- **POPULATIONS** : 4 populations scriptées.
- **Modèle nul** : `uniform_random`.
- **Contrôle négatif** : `always_a` — vérifié **au §6**.
- **H0 band** : ±0.1.

### Écarts v2 (§10)

- **max_tokens par famille (divergence au 16d)** : le gel v2 fixe
  `max_tokens = {glm: 2000, qwen: 400}` avant la vague définitive du jumeau Python. Le budget
  glm est un choix conservatif pour un modèle raisonneur ; ce carnet ne prétend
  pas avoir mesuré le plus petit budget suffisant. Température, parseur,
  politique réseau et seeds restent ceux du protocole du 16d.
- **Témoins de la politique réseau** : les 429, 500 et exceptions sont rejoués
  hors ligne avec transport et horloge simulés. Un échec persistant produit
  trois tentatives simulées et les délais simulés de 5 s puis 15 s avant
  `erreur_reseau`, sans reformulation.
- **Seeds incomplets** : H1 exige 5/5 seeds exploitables. H0 reste possible dès
  que son critère est observé ; sinon le verdict est `INCONCLUSIF` lorsqu'il
  reste des seeds exploitables, jamais H1 sur un sous-échantillon présenté comme complet.
- **Appariement exact des thetas** : le flux est dérivé du seed **seul**,
  identique des deux côtés, avec assertion d'égalité des séquences sur les 5 seeds.
  Pour `mix-dominant`, les écarts cités sont **exactement nuls**
  (+0.000 × 5 seeds), signature de thetas identiques + actions identiques.
- **qwen3.8-max : 429 persistants** — tous les appels de cette famille (120/120)
  sortent en `erreur_reseau` dans l'expérience Python citée (politique gelée :
  exclus des taux, comptés à part). Les deux verdicts qwen portent
  `EXCLU (aucune réponse exploitable)`.

Les tentatives HTTP, les tokens et les latences de l'expérience Python restent
historiques et cités : le rejeu C# n'accède à aucun fournisseur et ne mesure
aucune latence réelle.

## Exercices

Les trois exercices prolongent le carnet sur ses propres leviers : une
distribution `theta` **stratifiée** (exercice 1), un modèle nul **corrélé**
(exercice 2), une politique `best_reply` (exercice 3).

Une extension v2 (§10, populations mixtes LLM × scripts) suit les exercices ;
les énoncés ci-dessous restent ceux de la v1 scriptée.

Chacun dispose de sa cellule d'exécution : le stub rend `null` tant qu'il n'est
pas complété (`// TODO étudiant`), et le carnet s'exécute de bout en bout sans
lui. Les valeurs de référence sont celles du carnet — gaps au §4, bande H0 au
§5, oracle au §1.

Ces exercices sont regroupés ici, et non répartis dans les sections : les
exercices 1 et 2 se vérifient contre `agg` (§4) et la bande H0 (§5), qui
n'existent qu'après le driver §3 ; les placer plus haut mettrait les stubs
avant la machinerie qu'ils consomment.

### Exercice 1 — *Distribution theta stratifiée*

Remplace `theta_uniform` par une distribution où `theta_row` est marginalement
50/50 et `theta_col` est conditionnellement corrélé à `theta_row` (e.g.,
P(theta_col = theta_row) = 0.7). Le contrôle négatif `always_a` reste-t-il
rejeté ? Pourquoi ?

In [10]:
// Exercice 1 — distribution theta stratifiee (a completer)
// Objectif : theta_row marginalement 50/50, theta_col correle a theta_row
//            (P(theta_col == theta_row) = 0.7).
(string Row, string Col)? theta_stratifiee(Random rng)
{
    // TODO étudiant
    return null;
}

var _theta_ex1 = theta_stratifiee(new Random(0));
Console.WriteLine("Exercice 1 a completer : theta_stratifiee a renvoye " + (_theta_ex1?.ToString() ?? "None"));

Exercice 1 a completer : theta_stratifiee a renvoye None


### Exercice 2 — *Modèle nul corrélé*

Construis une population `biased` où les rapports `(r_row, r_col)` ont des
marginales uniformes et une corrélation de 0.7. Compare son welfare à
`uniform_random` — l'écart reste-t-il dans la bande H0 ?

In [11]:
// Exercice 2 — modele nul correle (a completer)
// Objectif : population `biased`, rapports (r_row, r_col) correles a 0.7.
(string Row, string Col)? rapports_correles(Random rng)
{
    // TODO étudiant
    return null;
}

var _rapports_ex2 = rapports_correles(new Random(0));
Console.WriteLine("Exercice 2 a completer : rapports_correles a renvoye " + (_rapports_ex2?.ToString() ?? "None"));

Exercice 2 a completer : rapports_correles a renvoye None


### Exercice 3 — *Nouvelle politique*

Définis une politique `best_reply` qui choisit `a'` si et seulement si le
rapport de l'adversaire est `a'` (et `a` sinon). Quel est son welfare attendu
sur ce mécanisme ? Justifie analytiquement puis vérifie
empiriquement (4 seeds suffisent ici).

In [12]:
// Exercice 3 — politique `best_reply` (a completer)
#nullable enable
string? rapport_best_reply(string r_adv)
{
    // TODO étudiant
    return null;
}

var _br_ex3 = rapport_best_reply("a'");
Console.WriteLine("Exercice 3 a completer : rapport_best_reply('a\\'') a renvoye " + (_br_ex3 ?? "None"));

Exercice 3 a completer : rapport_best_reply('a\'') a renvoye None


## 10. Populations mixtes LLM × scripts — la v2 hors ligne

La v1 ci-dessus joue des populations **100 % scriptées**. La v2 rejoue deux populations
hétérogènes qui mélangent agents LLM et agents scripts, à partir des données
citées du jumeau Python.

**Design.** Dans chaque paire de l'expérience Python, l'agent `row` est un LLM (condition
`llm-regles-visibles` du 16d : règles + tableau d'utilités
+ règle annoncée), l'agent `col` reste scripté et gelé :

| Population mixte | Politique `col` (gelée) | Ancre pure comparée (mêmes seeds) |
|---|---|---|
| `mix-dominant` | `a'` systématique (conforme) | `always_a_prime` (§2) |
| `mix-byzantin` | complément du type vrai (byzantin) | `byzantine` (§2) |

Chaque comparaison est **appariée par seed** : l'ancre pure est rejouée avec le
même `n_paires` v2 (12) et le même modèle de types `theta_uniform`, de sorte que
l'écart de welfare isole le coût (ou le gain) de remplacer un côté scripté par un
agent LLM. Les hypothèses sont **pré-enregistrées au §10.0 avant tout rejeu**,
et les deux issues de `mix-byzantin` sont publiables (résultat négatif accepté).

In [13]:
// §10.0 — Gel du protocole v2 (PREREG_V2, avant tout rejeu mixte)
// Au moins 2 populations hétérogènes mêlant agents LLM et agents scripts.
// Dans l'expérience citée, l'agent row est un LLM, l'agent col reste scripté.
var populationsMixtesV2 = Array.AsReadOnly(new[] { "mix-dominant", "mix-byzantin" });

// Gel des politiques du côté scripté et des ancres correspondantes.
var politiquesColV2 = new System.Collections.ObjectModel.ReadOnlyDictionary<string, (string Politique, string Ancre)>(
    new Dictionary<string, (string Politique, string Ancre)>(StringComparer.Ordinal)
    {
        ["mix-dominant"] = ("a' systematique (conforme)", "always_a_prime"),
        ["mix-byzantin"] = ("complement du type vrai (byzantin)", "byzantine")
    });
const string agentLLMV2 = "row, condition llm-regles-visibles (regles + tableau d'utilites + regle annoncee)";

// Familles réelles de l'expérience Python citée, dans l'ordre du protocole.
var famillesV2 = Array.AsReadOnly(new[] { (Cle: "glm", Modele: "glm-5.2"), (Cle: "qwen", Modele: "qwen3.8-max") });
// Gels d'appel repris du pilote, sauf le budget de tokens : 2000 pour glm est
// un choix conservatif pour un modèle raisonneur, pas un seuil minimal mesuré.
// Budget par famille gelé avant la vague définitive de l'expérience citée.
const double temperatureV2 = 0.7;
var budgetsTokensV2 = new System.Collections.ObjectModel.ReadOnlyDictionary<string, int>(
    new Dictionary<string, int>(StringComparer.Ordinal) { ["glm"] = 2000, ["qwen"] = 400 });
const int timeoutV2 = 120;
const int maxRetryParseV2 = 1; // 1 reformulation si JSON invalide, puis 'invalide'.
var backoffsV2 = Array.AsReadOnly(new[] { 0, 5, 15 });
const string politiqueReseauV2 = "timeout/erreur/5xx/429 : deux réessais ; après échec, erreur_reseau, exclue des taux et comptée à part";
const int concurrenceV2 = 8;

// Échantillon v2.
const int nPairesV2 = 12; // Par (population mixte, famille, seed).
var seedsV2 = Array.AsReadOnly(new[] { 0, 1, 7, 42, 99 }); // = SEEDS du gel §0 (assertion ci-dessous).
// Budget : 2 populations × 2 familles × 5 graines × 12 paires = 240 appels planifiés.
// Le plafond gelé de 320 porte sur les appels planifiés. Les tentatives HTTP
// historiques, reprises et reformulations comprises, sont comptées séparément :
// elles peuvent dépasser le plafond sans violer le gel.
const int budgetAppelsV2 = 320;

// Hypothèses pré-enregistrées : les deux issues sont publiables.
const double seuilTauxAprimeV2 = 0.8; // Seuil d'adoption de l'action dominante.
const double bandeWelfareV2 = 0.1; // = H0_NULL_BAND du gel §0 (assertion ci-dessous).
// Un verdict H1 exige les 5 seeds exploitables. Si des seeds manquent, H0
// reste possible dès que son critère est observé ; sinon INCONCLUSIF.
const string verdictsSeedsIncompletsV2 = "H0 dès son critère observé ; H1 seulement si les 5 graines sont exploitables et satisfont son critère ; sinon INCONCLUSIF en présence de données partielles.";
var hypothesesV2 = new System.Collections.ObjectModel.ReadOnlyDictionary<string, (string H0, string H1, Func<double, bool> CritereH0)>(
    new Dictionary<string, (string H0, string H1, Func<double, bool> CritereH0)>(StringComparer.Ordinal)
    {
        ["mix-dominant"] = ("Écart à always_a_prime < -bande sur au moins une graine", "Écart à always_a_prime >= -bande sur les cinq graines", d => d < -bandeWelfareV2),
        ["mix-byzantin"] = ("Écart à byzantine <= bande sur au moins une graine", "Écart à byzantine > bande sur les cinq graines", d => d <= bandeWelfareV2)
    });

var PREREG_V2 = new
{
    PopulationsMixtes = populationsMixtesV2, PolitiqueCol = politiquesColV2, AgentLLM = agentLLMV2,
    Familles = famillesV2, Temperature = temperatureV2, MaxTokens = budgetsTokensV2,
    TimeoutRequeteS = timeoutV2, MaxRetryParse = maxRetryParseV2, PolitiqueReseau = politiqueReseauV2,
    BackoffsS = backoffsV2, Concurrence = concurrenceV2, NPaires = nPairesV2, Seeds = seedsV2,
    BudgetAppelsMax = budgetAppelsV2, SeuilTauxAprime = seuilTauxAprimeV2, BandeWelfare = bandeWelfareV2,
    VerdictsSeedsIncomplets = verdictsSeedsIncompletsV2, Hyp = hypothesesV2
};
int appelsPlanifiesV2 = PREREG_V2.PopulationsMixtes.Count * PREREG_V2.Familles.Count * PREREG_V2.Seeds.Count * PREREG_V2.NPaires;

// Auto-cohérence du gel avant tout rejeu ; aucun appel LLM ni sonde de budget en C#.
bool ValiderGelV2()
{
    ExigerGelV1();
    Exiger(PREREG_V2.Seeds.SequenceEqual(SEEDS), "§10.0 : les graines v2 doivent être celles de la v1.");
    VerifierEcart("§10.0", "bande de welfare v2", PREREG_V2.BandeWelfare, H0_NULL_BAND, TOL_EXACTE);
    Exiger(PREREG_V2.PopulationsMixtes.SequenceEqual(new[] { "mix-dominant", "mix-byzantin" }), "§10.0 : populations mixtes et ordre du gel.");
    var familles = PREREG_V2.Familles.Select(f => f.Cle).ToArray();
    Exiger(PREREG_V2.Familles.SequenceEqual(new[] { ("glm", "glm-5.2"), ("qwen", "qwen3.8-max") }), "§10.0 : familles et modèles du gel.");
    Exiger(new HashSet<string>(familles, StringComparer.Ordinal).SetEquals(PREREG_V2.MaxTokens.Keys), "§10.0 : un budget de tokens par famille.");
    Exiger(new HashSet<string>(PREREG_V2.PopulationsMixtes, StringComparer.Ordinal).SetEquals(PREREG_V2.PolitiqueCol.Keys) &&
           new HashSet<string>(PREREG_V2.PopulationsMixtes, StringComparer.Ordinal).SetEquals(PREREG_V2.Hyp.Keys),
        "§10.0 : une politique et des hypothèses par population mixte.");
    Exiger(PREREG_V2.PolitiqueCol["mix-dominant"].Ancre == "always_a_prime" && PREREG_V2.PolitiqueCol["mix-byzantin"].Ancre == "byzantine",
        "§10.0 : ancres du gel.");
    Exiger(PREREG_V2.BackoffsS.SequenceEqual(new[] { 0, 5, 15 }), "§10.0 : calendrier des tentatives.");
    var dominantH0 = PREREG_V2.Hyp["mix-dominant"].CritereH0;
    var byzantinH0 = PREREG_V2.Hyp["mix-byzantin"].CritereH0;
    double bande = PREREG_V2.BandeWelfare;
    Exiger(dominantH0(-2 * bande) && !dominantH0(-bande) && byzantinH0(bande) && !byzantinH0(2 * bande),
        "§10.0 : les inclusions des bornes H0 diffèrent entre les deux populations.");
    VerifierEcart("§10.0", "populations mixtes", PREREG_V2.PopulationsMixtes.Count, 2, TOL_EXACTE);
    VerifierEcart("§10.0", "familles", PREREG_V2.Familles.Count, 2, TOL_EXACTE);
    VerifierEcart("§10.0", "graines", PREREG_V2.Seeds.Count, 5, TOL_EXACTE);
    VerifierEcart("§10.0", "paires par lot", PREREG_V2.NPaires, 12, TOL_EXACTE);
    VerifierEcart("§10.0", "max_tokens glm", PREREG_V2.MaxTokens["glm"], 2000, TOL_EXACTE);
    VerifierEcart("§10.0", "max_tokens qwen", PREREG_V2.MaxTokens["qwen"], 400, TOL_EXACTE);
    VerifierEcart("§10.0", "température", PREREG_V2.Temperature, 0.7, TOL_EXACTE);
    VerifierEcart("§10.0", "timeout historique", PREREG_V2.TimeoutRequeteS, 120, TOL_EXACTE);
    VerifierEcart("§10.0", "reformulations", PREREG_V2.MaxRetryParse, 1, TOL_EXACTE);
    VerifierEcart("§10.0", "concurrence historique", PREREG_V2.Concurrence, 8, TOL_EXACTE);
    VerifierEcart("§10.0", "seuil d'adoption de a'", PREREG_V2.SeuilTauxAprime, 0.8, TOL_EXACTE);
    VerifierEcart("§10.0", "appels planifiés", appelsPlanifiesV2, 240, TOL_EXACTE);
    VerifierEcart("§10.0", "budget d'appels", PREREG_V2.BudgetAppelsMax, 320, TOL_EXACTE);
    Exiger(appelsPlanifiesV2 <= PREREG_V2.BudgetAppelsMax, "§10.0 : budget respecté à la planification.");
    return true;
}

bool protocoleV2Gele = ValiderGelV2();
void ExigerGelV2() => Exiger(protocoleV2Gele, "Le gel v2 doit être vérifié avant le rejeu mixte.");
ExigerGelV2();
string texteBudgetsV2 = "{" + string.Join(", ", PREREG_V2.Familles.Select(f => Inv($"'{f.Cle}': {PREREG_V2.MaxTokens[f.Cle]}"))) + "}";
Console.WriteLine(Inv($"§10.0 — PREREG_V2 gele : {PREREG_V2.PopulationsMixtes.Count} populations mixtes, {PREREG_V2.NPaires} paires x {PREREG_V2.Familles.Count} familles x {PREREG_V2.Seeds.Count} seeds = {appelsPlanifiesV2} appels planifies / budget {PREREG_V2.BudgetAppelsMax} ; max_tokens par famille : {texteBudgetsV2}"));

§10.0 — PREREG_V2 gele : 2 populations mixtes, 12 paires x 2 familles x 5 seeds = 240 appels planifies / budget 320 ; max_tokens par famille : {'glm': 2000, 'qwen': 400}


### 10.1 Plomberie LLM — observation et validation reprises du 16d

`observation` et `valider_action` reprennent le schéma du 16d (cellule
§4 « Observation provider-neutral » de
`GameTheory-16d-LLM-Players-Othman-Sandholm-Python.ipynb`). Les
paramètres historiques de `PREREG_V2` sont : température 0.7, `max_tokens` **par famille**
(glm 2000, qwen 400), timeout 120 s, une reformulation de parse et trois
tentatives réseau pour l'appel en cours (5 s puis 15 s entre tentatives).
Trois échecs réseau retournent `erreur_reseau` sans reformulation.

Les exceptions réseau, les 429 et les 5xx suivent tous cette même boucle. Des
fixtures hors ligne exécutent `appel_llm_simule` avec un transport et une
horloge simulés : elles vérifient les délais, l'épuisement en `erreur_reseau`
et la reprise sur un 200 après un 429, sans appel fournisseur. « Cle API chargee »
est une sortie historique citée du jumeau Python ; aucune clé n'est lue en C#.

In [14]:
// §10.1 — Plomberie LLM (re-import 16e ; gels lus dans PREREG_V2)
#nullable enable
ExigerGelV2();
const string PROMPT_SYSTEM = "Tu es un agent rationnel dans une experience de theorie des mecanismes. Reponds STRICTEMENT en JSON, sans aucun autre texte.";
const string formatActionV2 = "repondre STRICTEMENT en JSON : {\"report\": \"a\"} ou {\"report\": \"a'\"}";
const string formatReformulationV2 = "REPRENDS : UNIQUEMENT le JSON {\"report\": \"a\"} ou {\"report\": \"a'\"}";
var ACTIONS_VALIDES = ACTIONS;

// L'observation est indépendante du fournisseur ; aucune permutation n'est jouée en v2.
Dictionary<string, object> observation(string role, string type_vrai, bool regles_visibles, bool permutation = false)
{
    var obs = new Dictionary<string, object>(StringComparer.Ordinal)
    {
        ["role"] = role, ["votre_type"] = type_vrai,
        ["espace_actions"] = ACTIONS_VALIDES.ToArray(), ["format_attendu"] = formatActionV2
    };
    if (regles_visibles)
    {
        string dominante_annoncee = permutation ? "a" : "a'";
        obs["regles"] = new Dictionary<string, object>(StringComparer.Ordinal)
        {
            ["mecanisme"] = "2 agents rapportent a ou a' ; l'outcome o1..o4 suit la matrice (a',a')->o1, (a',a)->o2, (a,a')->o3, (a,a)->o4",
            ["utilites"] = "les votres dependent de l'outcome ET de votre type (tableau fourni)",
            ["tableau_utilites"] = role == "row" ? u_row : u_col,
            ["regle_annoncee"] = $"l'action dominante est {dominante_annoncee} (rapportez {dominante_annoncee} quels que soient votre type et le rapport adverse)"
        };
    }
    return obs;
}

var optionsPromptV2 = new JsonSerializerOptions { Encoder = System.Text.Encodings.Web.JavaScriptEncoder.UnsafeRelaxedJsonEscaping };
string prompt_utilisateur(IReadOnlyDictionary<string, object> obs)
{
    string espace = "[" + string.Join(", ", ((string[])obs["espace_actions"]).Select(a => a == "a'" ? "\"a'\"" : "'" + a + "'")) + "]";
    string debut = $"Role : {obs["role"]}. Ton type (prive) : {obs["votre_type"]}\nEspace d'actions : {espace}\n";
    return obs.TryGetValue("regles", out var regles)
        ? debut + $"Regles du mecanisme : {JsonSerializer.Serialize(regles, optionsPromptV2)}\nObjectif : maximiser ton utilite (elle depend de l'outcome ET de ton type). {obs["format_attendu"]}"
        : debut + $"Aucune regle ne t'est communiquée. Choisis une action. {obs["format_attendu"]}";
}

(string? Action, string Statut) valider_action(string? brut)
{
    if (brut is null)
        return (null, "invalide");
    try
    {
        using var doc = JsonDocument.Parse(brut);
        if (doc.RootElement.ValueKind == JsonValueKind.Object &&
            doc.RootElement.TryGetProperty("report", out var report) && report.ValueKind == JsonValueKind.String &&
            report.GetString() is string action && ACTIONS_VALIDES.Contains(action))
            return (action, "valide");
    }
    catch (JsonException)
    {
        return (null, "invalide");
    }
    return (null, "invalide");
}

// Auto-test du schema (deterministe, sans LLM) — mêmes cas qu'en Python.
Exiger(valider_action(JsonSerializer.Serialize(new { report = "a'" })) == ("a'", "valide"), "§10.1 : action dominante valide.");
Exiger(valider_action(JsonSerializer.Serialize(new { report = "b" })) == (null, "invalide"), "§10.1 : action hors domaine.");
Exiger(valider_action("je rapporte a") == (null, "invalide"), "§10.1 : réponse non JSON.");
Exiger(valider_action(null) == (null, "invalide"), "§10.1 : réponse absente.");
var obsVisibleV2 = observation("row", "a", true);
var obsMasqueeV2 = observation("row", "a", false);
Exiger(obsVisibleV2.ContainsKey("regles") && !obsMasqueeV2.ContainsKey("regles") &&
    prompt_utilisateur(obsVisibleV2).Contains("l'action dominante est a'", StringComparison.Ordinal),
    "§10.1 : contenu du schéma d'observation.");

// Chaque événement est une fixture locale : aucun client HTTP, aucune clé, aucune attente réelle.
(string? Action, string Statut, int Tokens, int Tentatives) appel_llm_simule(
    string famille, string modele, Dictionary<string, object> obs, int seed,
    Func<string, (int Code, string? Brut, int Tokens)> transportSimule, Action<int> horlogeSimulee)
{
    ExigerGelV2();
    int tentatives = 0, tokensTotal = 0;
    var observationCourante = new Dictionary<string, object>(obs, StringComparer.Ordinal);
    for (int essai = 0; essai <= PREREG_V2.MaxRetryParse; essai++)
    {
        string corps = JsonSerializer.Serialize(new
        {
            model = modele, temperature = PREREG_V2.Temperature, max_tokens = PREREG_V2.MaxTokens[famille], seed,
            messages = new[] { new { role = "system", content = PROMPT_SYSTEM }, new { role = "user", content = prompt_utilisateur(observationCourante) } }
        });
        (int Code, string? Brut, int Tokens)? reponse = null;
        foreach (int backoff in PREREG_V2.BackoffsS) // Politique réseau gelée.
        {
            if (backoff != 0)
                horlogeSimulee(backoff);
            // Compte chaque tentative simulée, reformulations et reprises incluses.
            tentatives++;
            (int Code, string? Brut, int Tokens) item;
            try { item = transportSimule(corps); }
            catch (System.IO.IOException) { continue; } // Exception simulée : nouvelle tentative dans CET essai.
            if (item.Code == 429 || item.Code >= 500)
                continue; // 429/5xx : même politique que pour une exception.
            reponse = item;
            break; // Réponse exploitable : sortir de la boucle réseau.
        }
        if (!reponse.HasValue)
            return (null, "erreur_reseau", tokensTotal, tentatives);
        var rep = reponse.Value;
        Exiger(rep.Code >= 200 && rep.Code < 300, Inv($"§10.1 : statut non exploitable dans la fixture : {rep.Code}."));
        tokensTotal += rep.Tokens;
        var (action, statut) = valider_action(rep.Brut);
        if (statut == "valide")
            return (action, statut, tokensTotal, tentatives);
        if (essai < PREREG_V2.MaxRetryParse) // Reformulation unique (gel).
            observationCourante["format_attendu"] = formatReformulationV2;
    }
    return (null, "invalide", tokensTotal, tentatives);
}

// Témoins déterministes : la même fonction de rejeu utilise un transport et une horloge simulés.
var temoinsReseauV2 = new (string Etiquette, int[] Codes, string Statut, int Tentatives, int[] Delais)[]
{
    ("429", new[] { 429, 429, 429 }, "erreur_reseau", 3, new[] { 5, 15 }),
    ("500", new[] { 500, 500, 500 }, "erreur_reseau", 3, new[] { 5, 15 }),
    ("'exception'", new[] { -1, -1, -1 }, "erreur_reseau", 3, new[] { 5, 15 }),
    ("'429 puis 200'", new[] { 429, 200 }, "valide", 2, new[] { 5 })
};
var texteTemoinsV2 = new List<string>();
var familleFixtureV2 = PREREG_V2.Familles[0];
foreach (var temoin in temoinsReseauV2)
{
    var codes = new Queue<int>(temoin.Codes);
    var delais = new List<int>();
    int appelsFixture = 0;
    var obsFixture = observation("row", "a", false);
    obsFixture["format_attendu"] = "repondre en JSON : {\"report\": \"a\"}";
    var resultat = appel_llm_simule(familleFixtureV2.Cle, familleFixtureV2.Modele, obsFixture, 0, _ =>
    {
        appelsFixture++;
        int code = codes.Dequeue();
        if (code == -1)
            throw new System.IO.IOException("fixture");
        return (code, "{\"report\": \"a'\"}", 0);
    }, delais.Add);
    Exiger(resultat.Statut == temoin.Statut && resultat.Action == (temoin.Statut == "valide" ? "a'" : null),
        $"§10.1 : issue inattendue pour {temoin.Etiquette}.");
    Exiger(resultat.Tentatives == appelsFixture && codes.Count == 0, $"§10.1 : consommation de la fixture {temoin.Etiquette}.");
    VerifierEcart("§10.1", $"tentatives simulées {temoin.Etiquette}", resultat.Tentatives, temoin.Tentatives, TOL_EXACTE);
    VerifierEcart("§10.1", $"nombre de délais {temoin.Etiquette}", delais.Count, temoin.Delais.Length, TOL_EXACTE);
    for (int i = 0; i < delais.Count; i++)
        VerifierEcart("§10.1", Inv($"délai {temoin.Etiquette} à l'indice {i}"), delais[i], temoin.Delais[i], TOL_EXACTE);
    string listeDelais = string.Join(", ", delais.Select(d => Inv($"{d}")));
    texteTemoinsV2.Add(Inv($"({temoin.Etiquette}, {resultat.Tentatives}, [{listeDelais}])"));
}

// Deux réponses invalides épuisent la reformulation sans déclencher de reprise réseau.
var formatsSimulesV2 = new List<string>();
var parseEpuiseV2 = appel_llm_simule(familleFixtureV2.Cle, familleFixtureV2.Modele, obsVisibleV2, 0, corps =>
{
    using var requete = JsonDocument.Parse(corps);
    formatsSimulesV2.Add(requete.RootElement.GetProperty("messages")[1].GetProperty("content").GetString() ?? "");
    return (200, "{\"report\": \"b\"}", 0);
}, _ => throw new InvalidOperationException("§10.1 : pas de reprise réseau sur JSON invalide avec un statut 200."));
Exiger(parseEpuiseV2.Statut == "invalide" && parseEpuiseV2.Action is null && parseEpuiseV2.Tentatives == 2 &&
    formatsSimulesV2.Count == 2 && formatsSimulesV2[0].EndsWith(formatActionV2, StringComparison.Ordinal) &&
    formatsSimulesV2[1].EndsWith(formatReformulationV2, StringComparison.Ordinal), "§10.1 : une seule reformulation sur action invalide.");

// Citation historique de la sortie Python, pas un état de l'environnement C#.
Console.WriteLine("[Python, cité] Cle API chargee");
Console.WriteLine("Temoins reseau (hors ligne) : [" + string.Join(", ", texteTemoinsV2) + "]");
string texteFamillesV2 = "{" + string.Join(", ", PREREG_V2.Familles.Select(f => $"'{f.Cle}': '{f.Modele}'")) + "}";
Console.WriteLine("Plomberie v2 prete ; familles configurees : " + texteFamillesV2);

[Python, cité] Cle API chargee


Temoins reseau (hors ligne) : [(429, 3, [5, 15]), (500, 3, [5, 15]), ('exception', 3, [5, 15]), ('429 puis 200', 2, [5])]


Plomberie v2 prete ; familles configurees : {'glm': 'glm-5.2', 'qwen': 'qwen3.8-max'}


### 10.2 Populations mixtes et ancres appariées — exécution multi-seed

Le driver `run_mixed` reprend le même `theta_uniform` que le §2 ; dans
l'expérience Python, l'appel LLM porte le seed dérivé `seed * 31 + i`.
**Appariement exact des thetas** : le flux des types est dérivé du seed **seul**
(`seed * 100003 + 97`), identique pour la population mixte et son ancre au même
seed — à seed donné, les mêmes types sont joués des deux côtés, l'écart de
welfare isole donc le remplacement d'un côté scripté par un LLM, pas le tirage
des types (une assertion du driver atteste l'égalité des séquences sur les 5
seeds). `run_ancre_v2` rejoue les ancres pures (`always_a_prime`, `byzantine`)
sur **12 paires** — la comparaison v2 est appariée graine par graine,
indépendante du §3 (50 paires en v1).

Le C# reconstitue les choix de profils avec le générateur du §3 ; il ne
dispose pas d'un journal de tirages à citer. Les actions et statuts sont
reconstruits à partir de deux faits exhaustifs **cités du jumeau Python** :
les 120 réponses glm sont toutes valides et rapportent `a'`, et les 120 appels
qwen sont tous en erreur réseau. Ce ne sont pas des réponses brutes. Le rejeu
parcourt les 20 tâches (2 populations × 2 familles × 5 seeds) dans l'ordre du
gel, puis chaque paire dans l'ordre de son indice. Un tour dont la réponse
LLM est invalide ou en erreur réseau est **marqué sans substitution**
(précédent 16d) : aucune action de repli n'est jouée à sa place.

In [15]:
// §10.2 — Driver mixte + ancres appariees, execution multi-seed
#nullable enable
ExigerGelV2();

// Flux theta PARTAGE : derive du seed SEUL, identique pour la population mixte et
// son ancre au meme seed — l'appariement est EXACT (memes thetas joues des deux
// cotes), l'ecart de welfare isole donc les actions, pas le tirage des types.
ChoixCPython _rng_theta(int seed) => new ChoixCPython(checked((uint)(seed * 100003 + 97)));

// Faits exhaustifs cités : sortie Python §10.2 (statuts), §10.3 et §10.4 (actions).
// Encodage par famille : aucun contenu brut, nombre de tokens ou délai individuel n'est disponible.
var reponsesHistoriquesV2 = new System.Collections.ObjectModel.ReadOnlyDictionary<string, (string? Action, string Statut)>(
    new Dictionary<string, (string? Action, string Statut)>(StringComparer.Ordinal)
    {
        ["glm"] = ("a'", "valide"),
        ["qwen"] = (null, "erreur_reseau")
    });
Exiger(new HashSet<string>(reponsesHistoriquesV2.Keys, StringComparer.Ordinal).SetEquals(PREREG_V2.Familles.Select(f => f.Cle)),
    "§10.2 : les données historiques doivent couvrir les familles gelées.");

string rapport_ancre_v2(string ancre, string theta) => ancre switch
{
    "always_a_prime" => r_always_aprime(),
    "byzantine" => r_byzantine(theta),
    _ => throw new ArgumentException("Ancre hors protocole.", nameof(ancre))
};

List<(int I, (string Row, string Col) Theta, int SeedLLM, string? RRowLLM, string StatutLLM, string RCol, string? Outcome, int? Sw, int SwOptimal)> run_mixed(string pop, string famille, int seed)
{
    ExigerGelV2();
    var rng = _rng_theta(seed);
    string ancre = PREREG_V2.PolitiqueCol[pop].Ancre;
    var (action, statut) = reponsesHistoriquesV2[famille];
    bool valide = statut == "valide";
    Exiger(valide ? action is string rapport && ACTIONS_VALIDES.Contains(rapport) : action is null,
        $"§10.2 : statut et rapport historique incohérents pour {famille}.");
    var rows = new List<(int I, (string Row, string Col) Theta, int SeedLLM, string? RRowLLM, string StatutLLM, string RCol, string? Outcome, int? Sw, int SwOptimal)>(PREREG_V2.NPaires);
    for (int i = 0; i < PREREG_V2.NPaires; i++)
    {
        var th = theta_uniform(rng.Suivant); // meme modele de types que §2, meme flux que l'ancre
        int seedLLM = checked(seed * 31 + i);
        string rCol = rapport_ancre_v2(ancre, th.Col); // byzantin : complement du type vrai du cote scripte
        // invalide / erreur reseau : tour marque, aucune substitution (16e)
        string? outcome = action is string rRow ? mecanisme(rRow, rCol) : null;
        int? sw = outcome is string issue ? sw_oracle(issue, th.Row, th.Col) : (int?)null;
        int optimum = sw_optimal(th.Row, th.Col);
        Exiger(!sw.HasValue || sw.Value <= optimum, "§10.2 : le welfare réalisé ne doit pas dépasser l'optimum.");
        rows.Add((i, th, seedLLM, action, statut, rCol, outcome, sw, optimum));
    }
    Exiger(rng.NombreDeChoix == PREREG_V2.NPaires, "§10.2 : un choix de profil par paire mixte.");
    return rows;
}

List<(int I, (string Row, string Col) Theta, string RRow, string RCol, string Outcome, int Sw, int SwOptimal)> run_ancre_v2(string ancre, int seed)
{
    ExigerGelV2();
    var rng = _rng_theta(seed);
    var rows = new List<(int I, (string Row, string Col) Theta, string RRow, string RCol, string Outcome, int Sw, int SwOptimal)>(PREREG_V2.NPaires);
    for (int i = 0; i < PREREG_V2.NPaires; i++)
    {
        var th = theta_uniform(rng.Suivant);
        string rRow = rapport_ancre_v2(ancre, th.Row);
        string rCol = rapport_ancre_v2(ancre, th.Col);
        string outcome = mecanisme(rRow, rCol);
        rows.Add((i, th, rRow, rCol, outcome, sw_oracle(outcome, th.Row, th.Col), sw_optimal(th.Row, th.Col)));
    }
    Exiger(rng.NombreDeChoix == PREREG_V2.NPaires, "§10.2 : un choix de profil par paire d'ancre.");
    return rows;
}

// Les fixtures du §10.1 n'ont émis aucune requête réelle ; le rejeu n'en émet pas non plus.
// La télémétrie de la vague Python est citée séparément, sans la ventiler par paire.
var taches_mixtes = (from pop in PREREG_V2.PopulationsMixtes
                     from famille in PREREG_V2.Familles
                     from seed in PREREG_V2.Seeds
                     select (Pop: pop, Famille: famille.Cle, Seed: seed)).ToArray();
// cle = (pop, famille, seed) — la famille et non le nom du modèle sert de clé.
var mixed_results = taches_mixtes.ToDictionary(t => t, t => run_mixed(t.Pop, t.Famille, t.Seed));
var lotsAncresV2 = (from pop in PREREG_V2.PopulationsMixtes
                    from seed in PREREG_V2.Seeds
                    select (Ancre: PREREG_V2.PolitiqueCol[pop].Ancre, Seed: seed)).ToArray();
var ancres_v2 = lotsAncresV2.ToDictionary(t => t, t => run_ancre_v2(t.Ancre, t.Seed));

// Temoin d'appariement exact (grade T) : a seed donne, la population mixte et son
// ancre ont joue les MEMES thetas — sinon l'ecart de welfare ne serait pas purement
// un effet d'action.
var grainesApparieesV2 = new HashSet<int>();
foreach (var t in taches_mixtes)
{
    var rows = mixed_results[t];
    var ancre = ancres_v2[(PREREG_V2.PolitiqueCol[t.Pop].Ancre, t.Seed)];
    Exiger(rows.Select(r => (r.I, r.Theta)).SequenceEqual(ancre.Select(r => (r.I, r.Theta))),
        Inv($"§10.2 : appariement exact requis pour {t.Pop}/{t.Famille}, seed {t.Seed}."));
    Exiger(rows.Select(r => r.RCol).SequenceEqual(ancre.Select(r => r.RCol)),
        Inv($"§10.2 : le côté scripté doit rester identique pour {t.Pop}/{t.Famille}, seed {t.Seed}."));
    Exiger(rows.Count == PREREG_V2.NPaires && ancre.Count == PREREG_V2.NPaires, "§10.2 : taille d'un lot différente du gel.");
    grainesApparieesV2.Add(t.Seed);
}

// Références citées : sortie de la cellule 41 du jumeau Python.
int n_paires_mixtes = taches_mixtes.Sum(t => mixed_results[t].Count);
var statutsV2 = taches_mixtes.SelectMany(t => mixed_results[t]).GroupBy(r => r.StatutLLM)
    .ToDictionary(g => g.Key, g => g.Count(), StringComparer.Ordinal);
VerifierEcart("§10.2", "tâches mixtes", taches_mixtes.Length, 20, TOL_EXACTE);
VerifierEcart("§10.2", "lots mixtes", mixed_results.Count, taches_mixtes.Length, TOL_EXACTE);
VerifierEcart("§10.2", "paires mixtes", n_paires_mixtes, 240, TOL_EXACTE);
VerifierEcart("§10.2", "lots d'ancres", ancres_v2.Count, 10, TOL_EXACTE);
VerifierEcart("§10.2", "paires par ancre", PREREG_V2.NPaires, 12, TOL_EXACTE);
VerifierEcart("§10.2", "graines appariées", grainesApparieesV2.Count, 5, TOL_EXACTE);
VerifierEcart("§10.2", "appels planifiés historiques", n_paires_mixtes, appelsPlanifiesV2, TOL_EXACTE);
VerifierEcart("§10.2", "budget planifié", PREREG_V2.BudgetAppelsMax, 320, TOL_EXACTE);
Exiger(statutsV2.Count == 2 && statutsV2.ContainsKey("valide") && statutsV2.ContainsKey("erreur_reseau"), "§10.2 : statuts historiques attendus.");
VerifierEcart("§10.2", "réponses valides reconstruites", statutsV2["valide"], 120, TOL_EXACTE);
VerifierEcart("§10.2", "erreurs réseau reconstruites", statutsV2["erreur_reseau"], 120, TOL_EXACTE);

// Télémétrie exclusivement citée : le rapport calculé utilise le total de durée publié à une décimale.
const int tentativesHttpCiteesV2 = 480;
const int tokensCitesV2 = 29355;
const double latenceCumuleeCiteeV2 = 1436.0;
double tolLatenceParTentativeV2 = TolerancePubliee(2) + TolerancePubliee(1) / tentativesHttpCiteesV2;
double latenceParTentativeV2 = latenceCumuleeCiteeV2 / tentativesHttpCiteesV2;
VerifierEcart("§10.2", "ratio de latence à partir de la télémétrie citée", latenceParTentativeV2, 2.99, tolLatenceParTentativeV2);
Exiger(Inv($"{latenceParTentativeV2:F2}") == "2.99", "§10.2 : précision publiée du ratio de latence.");

Console.WriteLine(Inv($"§10.2 — {taches_mixtes.Length} taches LLM, {n_paires_mixtes} paires mixtes rejouees ({ancres_v2.Count} ancres pures x {PREREG_V2.NPaires} paires rejouees)"));
Console.WriteLine(Inv($"  Appariement exact des thetas verifie (assert) sur les {grainesApparieesV2.Count} seeds."));
Console.WriteLine(Inv($"  Appels planifies : {n_paires_mixtes} / budget {PREREG_V2.BudgetAppelsMax} ; tentatives HTTP reelles [Python, cité] : {tentativesHttpCiteesV2}"));
Console.WriteLine(Inv($"  Statuts LLM [reconstruits] : {{'valide': {statutsV2["valide"]}, 'erreur_reseau': {statutsV2["erreur_reseau"]}}}"));
Console.WriteLine(Inv($"  Tokens totaux [Python, cité] : {tokensCitesV2} ; latence cumulee [Python, citée] : {latenceCumuleeCiteeV2:F1} s ({latenceParTentativeV2:F2} s/tentative, calcul sur les totaux cités)"));

§10.2 — 20 taches LLM, 240 paires mixtes rejouees (10 ancres pures x 12 paires rejouees)


  Appariement exact des thetas verifie (assert) sur les 5 seeds.


  Appels planifies : 240 / budget 320 ; tentatives HTTP reelles [Python, cité] : 480


  Statuts LLM [reconstruits] : {'valide': 120, 'erreur_reseau': 120}


  Tokens totaux [Python, cité] : 29355 ; latence cumulee [Python, citée] : 1436.0 s (2.99 s/tentative, calcul sur les totaux cités)


### 10.3 Agrégation et verdicts H0/H1 v2

Pour chaque `(population mixte, famille, seed)` : taux de `a'` côté LLM sur les
réponses **valides**, effectifs par statut et welfare moyen. Le verdict applique
les seuils gelés au §10.0 à la différence **appariée** avec l'ancre (même seed,
mêmes thetas et même `n_paires`). Dans les données citées, chaque seed est
soit entièrement exploitable, soit entièrement exclu : la moyenne mixte est
calculée sur les réponses valides et la moyenne de l'ancre sur ses 12 paires.

**H1 exige les 5 seeds exploitables** et le respect de son critère sur chacun.
Un seed sans réponse exploitable ne devient jamais un choix du modèle : si le
critère de **H0** est observé sur un seed disponible, H0 reste possible ; sinon, des
seeds manquants rendent le verdict **INCONCLUSIF**, avec les effectifs affichés.
Sans aucune réponse exploitable, la famille est **EXCLUE**. Cette règle lève
l'ambiguïté entre les hypothèses du §10.0 et l'exclusion réseau.

In [16]:
// §10.3 — Agregation mixte + verdicts (hypotheses gelees au §10.0)
ExigerGelV2();
// Les synthèses Python sont publiées à deux décimales, sauf l'écart moyen (trois).
// Ces tolérances s'appliquent aux références arrondies, jamais aux verdicts.
double tolDeuxDecimalesV2 = TolerancePubliee(2);
double tolTroisDecimalesV2 = TolerancePubliee(3);
const string verdictExcluV2 = "EXCLU (aucune reponse exploitable)";
string Signe3V2(double x) => Inv($"{x:+0.000;-0.000;+0.000}");
string ListeSeedsV2(IEnumerable<int> graines) => "[" + string.Join(", ", graines.Select(s => Inv($"{s}"))) + "]";
double? MoyenneDisponibleV2(IEnumerable<double?> valeurs)
{
    var disponibles = valeurs.Where(v => v.HasValue).Select(v => v.GetValueOrDefault()).ToArray();
    return disponibles.Length == 0 ? (double?)null : disponibles.Average();
}

var mixed_agg = new Dictionary<(string Pop, string Famille, int Seed), (double? TauxAprime, int NValide, int NInvalide, int NErreurReseau, double? SwMean)>();
foreach (var t in taches_mixtes)
{
    var rows = mixed_results[t];
    var valides = rows.Where(r => r.StatutLLM == "valide").ToArray();
    var sws = rows.Where(r => r.Sw.HasValue).Select(r => r.Sw.GetValueOrDefault()).ToArray();
    int nInvalide = rows.Count(r => r.StatutLLM == "invalide");
    int nErreurReseau = rows.Count(r => r.StatutLLM == "erreur_reseau");
    Exiger(rows.All(r => (r.StatutLLM == "valide") == r.Sw.HasValue) &&
        valides.Length + nInvalide + nErreurReseau == rows.Count,
        Inv($"§10.3 : les statuts et valeurs disponibles doivent coïncider pour {t.Pop}/{t.Famille}, seed {t.Seed}."));
    double? taux = valides.Length == 0 ? (double?)null : (double)valides.Count(r => r.RRowLLM == "a'") / valides.Length;
    double? swMean = sws.Length == 0 ? (double?)null : sws.Average();
    // Contrôle par lot dérivé des faits exhaustifs cités aux §10.2–10.4 du Python.
    int validesCites = t.Famille == "glm" ? PREREG_V2.NPaires : 0;
    string lot = Inv($"{t.Pop}/{t.Famille}, seed {t.Seed}");
    VerifierEcart("§10.3", $"réponses valides {lot}", valides.Length, validesCites, TOL_EXACTE);
    VerifierEcart("§10.3", $"réponses invalides {lot}", nInvalide, 0, TOL_EXACTE);
    VerifierEcart("§10.3", $"erreurs réseau {lot}", nErreurReseau, PREREG_V2.NPaires - validesCites, TOL_EXACTE);
    if (taux is double tx)
        VerifierEcart("§10.3", $"taux de a' {lot}", tx, 1.00, TOL_EXACTE);
    mixed_agg.Add(t, (taux, valides.Length, nInvalide, nErreurReseau, swMean));
}
var ancre_agg_v2 = lotsAncresV2.ToDictionary(t => t, t => ancres_v2[t].Average(r => r.Sw));

string ClasserV2(string pop, IReadOnlyCollection<double> ecarts)
{
    if (ecarts.Count == 0)
        return verdictExcluV2;
    if (ecarts.Any(PREREG_V2.Hyp[pop].CritereH0))
        return "H0";
    return ecarts.Count == PREREG_V2.Seeds.Count ? "H1" :
        Inv($"INCONCLUSIF ({ecarts.Count}/{PREREG_V2.Seeds.Count} seeds exploitables)");
}
// Cas limites de la règle gelée, indépendants des observations historiques.
double bandeTestV2 = PREREG_V2.BandeWelfare;
Exiger(ClasserV2("mix-dominant", new[] { -2 * bandeTestV2 }) == "H0" &&
       ClasserV2("mix-dominant", new[] { -bandeTestV2 }) == "INCONCLUSIF (1/5 seeds exploitables)" &&
       ClasserV2("mix-byzantin", new[] { bandeTestV2 }) == "H0" &&
       ClasserV2("mix-byzantin", new[] { 2 * bandeTestV2 }) == "INCONCLUSIF (1/5 seeds exploitables)" &&
       ClasserV2("mix-byzantin", Array.Empty<double>()) == verdictExcluV2,
    "§10.3 : seuils et verdicts en présence de graines manquantes.");

// Références citées : tableau et détail de la sortie de la cellule 43 du jumeau Python.
var syntheseCiteeV2 = new Dictionary<(string Pop, string Famille), (double? Taux, double? Mix, double Ancre, double? Diff, string Verdict, int Exclus)>
{
    [("mix-dominant", "glm")] = (1.00, 4.65, 4.65, 0.000, "H1", 0),
    [("mix-dominant", "qwen")] = (null, null, 4.65, null, verdictExcluV2, 5),
    [("mix-byzantin", "glm")] = (1.00, 3.82, 2.00, 1.817, "H1", 0),
    [("mix-byzantin", "qwen")] = (null, null, 2.00, null, verdictExcluV2, 5)
};
var diffsCiteesV2 = new Dictionary<string, double[]>(StringComparer.Ordinal)
{
    ["mix-dominant"] = new[] { 0.00, 0.00, 0.00, 0.00, 0.00 },
    ["mix-byzantin"] = new[] { 1.08, 3.17, 1.58, 0.92, 2.33 }
};
void VerifierDisponibleV2(string mesure, double? calcule, double? cite, double tolerance, Func<double, string> format)
{
    Exiger(calcule.HasValue == cite.HasValue, $"§10.3 : disponibilité différente pour {mesure}.");
    if (calcule is double valeur && cite is double reference)
    {
        VerifierEcart("§10.3", mesure, valeur, reference, tolerance);
        Exiger(format(valeur) == format(reference), $"§10.3 : valeur affichée différente pour {mesure}.");
    }
}

var groupesV2 = (from pop in PREREG_V2.PopulationsMixtes from f in PREREG_V2.Familles select (Pop: pop, Famille: f.Cle)).ToArray();
var verdicts_v2 = new Dictionary<(string Pop, string Famille), (string Ancre, string Verdict, List<(int Seed, double Diff)> Diffs, double? MeanDiff, List<int> SeedsExclus, double? TauxMoy, double? SwMixte, double SwAncre)>();
foreach (var groupe in groupesV2)
{
    var (pop, famille) = groupe;
    string ancre = PREREG_V2.PolitiqueCol[pop].Ancre;
    var ecartsParSeed = new List<(int Seed, double Diff)>();
    var seedsExclus = new List<int>();
    foreach (int s in PREREG_V2.Seeds)
    {
        var a = mixed_agg[(pop, famille, s)];
        if (a.SwMean is not double sw)
        {
            seedsExclus.Add(s); // aucune reponse exploitable : exclu, rapporte
            continue;
        }
        ecartsParSeed.Add((s, sw - ancre_agg_v2[(ancre, s)]));
    }
    var ecarts = ecartsParSeed.Select(e => e.Diff).ToArray();
    string verdict = ClasserV2(pop, ecarts);
    double? meanDiff = ecarts.Length == 0 ? (double?)null : ecarts.Average();
    double? tauxMoy = MoyenneDisponibleV2(PREREG_V2.Seeds.Select(s => mixed_agg[(pop, famille, s)].TauxAprime));
    double? swMixte = MoyenneDisponibleV2(PREREG_V2.Seeds.Select(s => mixed_agg[(pop, famille, s)].SwMean));
    double swAncre = PREREG_V2.Seeds.Average(s => ancre_agg_v2[(ancre, s)]);
    var reference = syntheseCiteeV2[groupe];
    string nom = $"{pop}/{famille}";
    Exiger(verdict == reference.Verdict, $"§10.3 : verdict différent pour {nom}.");
    VerifierEcart("§10.3", $"graines exclues {nom}", seedsExclus.Count, reference.Exclus, TOL_EXACTE);
    VerifierDisponibleV2($"taux moyen {nom}", tauxMoy, reference.Taux, TOL_EXACTE, x => Inv($"{x:F2}"));
    VerifierDisponibleV2($"SW mixte {nom}", swMixte, reference.Mix, tolDeuxDecimalesV2, x => Inv($"{x:F2}"));
    VerifierDisponibleV2($"SW ancre {nom}", swAncre, reference.Ancre, tolDeuxDecimalesV2, x => Inv($"{x:F2}"));
    VerifierDisponibleV2($"écart moyen {nom}", meanDiff, reference.Diff, tolTroisDecimalesV2, Signe3V2);
    if (famille == "glm")
    {
        Exiger(ecartsParSeed.Select(e => e.Seed).SequenceEqual(PREREG_V2.Seeds), $"§10.3 : ordre des graines exploitables pour {nom}.");
        for (int i = 0; i < ecartsParSeed.Count; i++)
            VerifierDisponibleV2(Inv($"écart {nom}, seed {ecartsParSeed[i].Seed}"), ecartsParSeed[i].Diff,
                diffsCiteesV2[pop][i], pop == "mix-dominant" ? TOL_EXACTE : tolDeuxDecimalesV2, Signe2);
    }
    else
        Exiger(seedsExclus.SequenceEqual(new[] { 0, 1, 7, 42, 99 }) && ecartsParSeed.Count == 0, $"§10.3 : graines exclues différentes pour {nom}.");
    verdicts_v2.Add(groupe, (ancre, verdict, ecartsParSeed, meanDiff, seedsExclus, tauxMoy, swMixte, swAncre));
}

var vDominantV2 = verdicts_v2[("mix-dominant", "glm")];
var vByzantinV2 = verdicts_v2[("mix-byzantin", "glm")];
Exiger(vByzantinV2.SwMixte is double swByzantin && swByzantin > vByzantinV2.SwAncre && swByzantin < vDominantV2.SwAncre,
    "§10.3 : l'amortissement byzantin doit être positif et partiel.");

Console.WriteLine("§10.3 — Synthese v2 par (population mixte, famille) :");
Console.WriteLine(Inv($"{"pop",-14} {"famille",-7} {"taux a' LLM",11} {"SW mixte",9} {"SW ancre",9} {"diff",7} {"verdict",-10} exclus"));
foreach (var groupe in groupesV2)
{
    var v = verdicts_v2[groupe];
    string taux = v.TauxMoy is double tx ? Inv($"{tx:F2}") : "n/a";
    string mixte = v.SwMixte is double sw ? Inv($"{sw:F2}") : "n/a";
    string diff = v.MeanDiff is double d ? Signe3V2(d) : "n/a";
    Console.WriteLine(Inv($"{groupe.Pop,-14} {groupe.Famille,-7} {taux,11} {mixte,9} {v.SwAncre,9:F2} {diff,7} {v.Verdict,-10} {v.SeedsExclus.Count}"));
}
Console.WriteLine();
Console.WriteLine("Detail des diffs apparies par seed (mixte - ancre) :");
foreach (var groupe in groupesV2)
{
    var v = verdicts_v2[groupe];
    string parSeed = string.Join(", ", v.Diffs.Select(e => Inv($"s{e.Seed}:{Signe2(e.Diff)}")));
    string exclus = v.SeedsExclus.Count == 0 ? "" : " ; exclus : " + ListeSeedsV2(v.SeedsExclus);
    Console.WriteLine($"  {groupe.Pop}/{groupe.Famille} vs {v.Ancre} : {parSeed}{exclus}");
}

// Attestation : vérifier chaque écart sous sa propre tolérance avant de résumer la section.
Console.WriteLine();
Console.WriteLine("Attestation de parité — contrôles numériques du rejeu C# :");
foreach (string section in new[] { "§0", "§1", "§3", "§4", "§5", "§6", "§7", "§10.0", "§10.1", "§10.2", "§10.3" })
{
    Exiger(ecartsParSection.ContainsKey(section) && ecartsParSection[section].Count > 0, $"Attestation manquante : {section}.");
    var mesures = ecartsParSection[section];
    Exiger(mesures.All(m => m.Ecart <= m.Tolerance), $"Écart hors tolérance : {section}.");
    var max = mesures.OrderByDescending(m => m.Ecart).First();
    Console.WriteLine(Inv($"  {section} : CONFORME, {mesures.Count} comparaisons ; écart maximal {max.Ecart:G6} ({max.Mesure}, tolérance {max.Tolerance:G6})."));
}
Console.WriteLine("  §2 : registre des stratégies conforme à la liste citée ; comparaisons de libellés et de verdicts vérifiées par assertion dans leurs sections.");
Console.WriteLine("  Profils tirés : flux CPython reconstitués, pas de journal de tirages cité.");
Console.WriteLine("  Actions et statuts LLM : reconstruits à partir des observations exhaustives citées, pas de nouvel appel.");
Console.WriteLine("  Citations seules, non mesurées ici : chargement historique de la clé, tentatives HTTP réelles, tokens et durée cumulée Python.");

§10.3 — Synthese v2 par (population mixte, famille) :


pop            famille taux a' LLM  SW mixte  SW ancre    diff verdict    exclus


mix-dominant   glm            1.00      4.65      4.65  +0.000 H1         0


mix-dominant   qwen            n/a       n/a      4.65     n/a EXCLU (aucune reponse exploitable) 5


mix-byzantin   glm            1.00      3.82      2.00  +1.817 H1         0


mix-byzantin   qwen            n/a       n/a      2.00     n/a EXCLU (aucune reponse exploitable) 5


... (17 outputs left out)

  §2 : registre des stratégies conforme à la liste citée ; comparaisons de libellés et de verdicts vérifiées par assertion dans leurs sections.


  Profils tirés : flux CPython reconstitués, pas de journal de tirages cité.


  Actions et statuts LLM : reconstruits à partir des observations exhaustives citées, pas de nouvel appel.


  Citations seules, non mesurées ici : chargement historique de la clé, tentatives HTTP réelles, tokens et durée cumulée Python.


### 10.4 Lecture des résultats v2

**Comportement LLM** : dans l'expérience Python citée, glm-5.2 rapporte l'action
dominante `a'` à taux **1.00** dans les deux populations mixtes (120/120 réponses
valides) — la règle visible est suivie quel que soit le partenaire scripté,
conforme ou byzantin.

**mix-dominant → H1, écart exactement nul** : la population mixte (LLM côté row,
`a'` systématique côté col) obtient le **même** welfare que son ancre
`always_a_prime` — diff **+0.000 sur chacun des 5 seeds**. Ce zéro exact est
informatif deux fois : (1) remplacer un côté scripté par un LLM qui suit la
règle dominante **ne coûte rien** en welfare moyen sur ces paires ; (2) il est
cohérent avec l'appariement exact — mêmes thetas (assertion du § 10.2) +
mêmes actions ⇒ même welfare, au dernier chiffre près.

**mix-byzantin → H1, amortissement partiel** : face à un partenaire byzantin
(qui rapporte le complément de son type vrai), le LLM **continue** de rapporter
`a'` (taux 1.00 — la stratégie dominante ne dépend pas du rapport adverse) et
la population mixte récupère **+1.82 de welfare moyen** sur la population
entièrement byzantine (3.82 vs 2.00 ; écarts par seed affichés de +0.92 à
+3.17). L'amortissement est **partiel** : 3.82 reste sous le niveau de
l'ancre dominante (4.65) — un seul côté conforme ne restaure pas tout le
welfare perdu par rapport à cette ancre (1.82 des 2.65 perdus, à deux décimales).

**qwen3.8-max** : exclusion réseau documentée dans la sortie Python citée
(429 persistants sur les 120 appels), pas une mesure du comportement — la
comparaison exploitable repose sur glm-5.2 seule. Les seeds exclus ne comptent
ni pour H0 ni pour H1 (politique gelée du 16d).

### 10.5 Portée et limites du grade B v2

- Le verdict v2 est établi sur les seeds {0, 1, 7, 42, 99} avec appariement
  exact des thetas (même n=12, même modèle de types, mêmes profils des deux
  côtés par seed) — pas au-delà.
- La famille qwen3.8-max est une exclusion réseau (429 persistants), pas une
  mesure : la comparaison exploitable repose sur glm-5.2 seule.
- Un pont B v2 héberge la défaillance de ses témoins : l'oracle du § 1,
  la bande ±0.1 gelée et l'assertion d'appariement des thetas. Toute relecture
  cite le seuil exact.
- Le comportement « rapporter a' » est observé dans l'expérience citée sur le
  prompt `llm-regles-visibles` du 16d ; la v2 ne teste ni la condition sans règles,
  ni la permutation (16d § 7).

## À retenir

Ce carnet généralise le pilote `GameTheory-16d` : là où le 16d jouait des **profils** fixes, le 16e joue des **populations** — une distribution sur le couple `(theta, r)` — et pose la question de la généralisation : le comportement mesuré tient-il quand on ne fixe plus le profil ?

**Ce que le protocole v1 établit.**

- **L'échantillon** : 4 populations × 5 seeds × 50 paires = 1000 paires jouées, sous un protocole gelé au §0 (seeds `[0, 1, 7, 42, 99]`, bande de nullité `± 0.1` autour du modèle nul `uniform_random`).
- **L'oracle** : le bien-être optimal par profil vrai vaut 4 sur `(a, a)` et 7 sur `(a', a')`, et la dominance stricte de `a'` en utilité individuelle est vérifiée par assertion.
- **Les écarts au nul** : le nul moyen vaut `3.8920`. La population `always_a_prime` s'en écarte de `+0.49` en moyenne, `always_a` de `−2.21` et `byzantine` de `−1.67` — soit **0 H0**, **3 H1** et 1 population de référence.
- **Le contrôle négatif est net** : l'écart `a' − a` vaut `+2.700` en moyenne, avec un minimum de `+2.000` et le signe correct sur les **5** seeds.

**Ce que la v2 ajoute (mixte LLM × scripts).**

- **Le protocole est gelé avant le rejeu** : 20 tâches LLM et 240 paires mixtes rejouées, pour 240 appels historiques planifiés dans un budget de 320. **Dans l'expérience Python citée**, les tentatives HTTP réelles, réessais gelés inclus, s'élèvent à 480, avec 120 réponses valides et 120 erreurs réseau, 29 355 tokens et 1436.0 s de latence cumulée. L'appariement exact des `theta` entre population mixte et ancre scriptée est vérifié par assertion sur les 5 seeds.
- **Un seul modèle fournit des réponses exploitables dans l'expérience citée** : `glm-5.2` renvoie un taux d'action `a'` de `1.00` ; `qwen3.8-max` est **exclu** — aucune réponse exploitable sur les 5 seeds, pour les deux populations.
- **Les verdicts v2** : sur la population mix-dominant, `glm` atteint un rendement de `4.65`, exactement celui de l'ancre scriptée, soit un écart de `+0.000` sur chacun des 5 seeds (H1) ; sur la population mix-byzantin, le rendement atteint `3.82` contre `2.00` pour l'ancre, un écart de `+1.817` (H1).

**Réserves maintenues.** Le taux `1.00` de `a'` pour `glm` mesure l'**obéissance à la règle**, pas une optimisation : sur la population mix-dominant, l'écart nul le montre — le LLM fait exactement ce que l'ancre scriptée fait, ni plus ni moins. L'exclusion de `qwen` est un fait d'**infrastructure** (statuts réseau), pas une propriété du modèle : elle interdit de conclure sur cette famille, non de la disqualifier. Enfin, la bande de nullité `± 0.1` est un **choix de seuil**, consigné au §8 : un seuil déplacé peut faire basculer une conclusion.

## Vérifiez votre compréhension

1. Le modèle nul `uniform_random` sert de référence. Pourquoi comparer un écart **au nul** plutôt que le rendement lui-même ?

   <details><summary>Réponse</summary>

   Parce que le rendement absolu dépend du profil tiré : l'oracle vaut 4 sur `(a, a)` et 7 sur `(a', a')`. Comparer au nul donne un point de référence sous la même loi de profils, mais ne supprime pas le bruit d'échantillonnage : les profils ne sont pas tous appariés en v1. La bande `± 0.1` porte sur l'écart par seed, pas sur le niveau ni sur la seule moyenne globale.

   </details>

2. La population `always_a` est un **contrôle négatif** : elle doit perdre. Que se passerait-il si elle gagnait ?

   <details><summary>Réponse</summary>

   Le contrôle échouerait. Sur les cinq échantillons gelés, le carnet vérifie que `always_a_prime` obtient un welfare moyen strictement supérieur à `always_a` sur des profils appariés. Si ce signe disparaît dans le rejeu, il faut vérifier les tables, les profils et les rapports avant d'interpréter les autres résultats. La dominance individuelle n'interdit pas un écart social négatif sur un profil particulier, comme le montre `(a, a)`.

   </details>

3. Sur la population mix-dominant, `glm` atteint `4.65`, exactement le rendement de l'ancre scriptée. L'écart `+0.000` est-il un échec du LLM ?

   <details><summary>Réponse</summary>

   Non : les mêmes rapports sur les mêmes profils donnent le même welfare. L'écart nul signifie que le remplacement ne dégrade pas le résultat de l'ancre, pas que l'ancre maximise le bien-être sur tous les profils. Sur la population mix-byzantin, où l'ancre est dégradée (`2.00`), l'écart de `+1.817` mesure le gain du même rapport dominant face à un partenaire byzantin.

   </details>

4. `qwen3.8-max` est exclu faute de réponses exploitables. Que peut-on en dire ?

   <details><summary>Réponse</summary>

   Quasiment rien sur le modèle, et beaucoup sur la conduite du protocole. Le jumeau Python consigne 120 appels qwen en erreur réseau dans une expérience totalisant 480 tentatives HTTP pour les deux familles. L'exclusion est donc un fait d'**infrastructure**, pas une propriété du modèle — et elle interdit d'écrire « qwen ne suit pas la règle », ce que rien dans les données ne soutient.

   </details>

## Pour aller plus loin

**Carnet suivant.** [GameTheory-16h — Coopération LLM](GameTheory-16h-Cooperation-LLM-Python.ipynb) -- la question voisine : le 16e mesure l'**incitation** d'un mécanisme (une action dominante face à une action dominée), le 16h mesure la **coopération** entre agents sous protocole multi-seed et hétérogénéités de population.

**Lectures.**

- La source du mécanisme est **Othman, S. & Sandholm, T. (2009)**, SAGT 2009, Proposition 6 (MOM) — dont la formalisation Lean est lue et citée dans le carnet `GameTheory-16` (§4.6) et dans `GameTheory-16d`.
- Les références complètes du carnet voisin `GameTheory-16h` listent **Axelrod (1984)**, **Nowak et Sigmund (1993)**, **Press et Dyson (2012)** et **Vallinder et Hughes (2024)** — le versant « coopération » de la même famille.

**Carnets voisins.**

- [GameTheory-16d — Joueurs LLM](GameTheory-16d-LLM-Players-Othman-Sandholm-Python.ipynb) -- le pilote dont ce carnet généralise le profil en population.
- [GameTheory-16 — Mécanismes](GameTheory-16-MechanismDesign-Python.ipynb) -- §4.6, la Proposition 6 et l'oracle de Shapley-Waldman.

> Source du protocole et des observations citées : [jumeau Python](GameTheory-16e-Populations-Othman-Sandholm-Python.ipynb).